# Papagun — самостоятельная генерация отчёта
Восемнадцать основных слайдов и 90 тестовых вариантов создаются программно. Вход — Excel; презентации не являются источником или шаблоном. Согласованные формулы сохранены. Логотип — отдельное изображение в resources.

Финансовые препараты: аналог слайда 6, financial_df, январь–сентябрь, тыс. тенге; сентябрьский отсутствующий факт по двум статьям равен нулю по согласованию. Итог финансового этапа — 5/25 слайдов.

Квартальные препараты из quantity_df: 1/2/3 кварталы и 9 месяцев, норма/факт 1С/разность, итоговый результат 6/30 слайдов.

Новые слайды 7–9 повторяют алгоритмы прежних слайдов 4–6 для статей «Эритропоэтин короткого действия» и «Сахарат железа (Венофер)(Аджифер)». Для них согласованы те же точечные правила нуля.

Слайды 10–12 повторяют алгоритмы для статей «Севеламера карбонат» и «Кальция ацетат (кальций ацтек)».

Слайды 13–15 выбирают статьи по кодам 211.07 и 211.10; подписи берутся из наименований соответствующей книги.

Слайды 16–18 используют коды 211.11 и 211.12; наименования для подписей берутся из своей книги.


## Зависимости выбранного ядра
Пакет называется `python-pptx`, а импортируется как `pptx`. Пакеты должны быть установлены в то Python-окружение, которое выбрано ядром notebook.

Если отсутствует `pptx`, выполните отдельную ячейку `%pip install python-pptx==1.0.2`, затем перезапустите ядро и выполните notebook сверху вниз. `%pip` устанавливает пакет в окружение текущего ядра.

Ниже проверяются три рабочие библиотеки. По умолчанию сеть не используется. Для установки отсутствующих пакетов этой же ячейкой установите `INSTALL_MISSING_DEPENDENCIES = True`, выполните её и при необходимости перезапустите ядро. Уже установленные пакеты не обновляются.

In [ ]:
import sys
import subprocess
import importlib.util
import importlib

INSTALL_MISSING_DEPENDENCIES = False
required_packages = {
    'pandas': 'pandas',
    'openpyxl': 'openpyxl',
    'pptx': 'python-pptx==1.0.2',
}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    install_command = '%pip install ' + ' '.join(missing_packages)
    if not INSTALL_MISSING_DEPENDENCIES:
        raise RuntimeError(
            'В выбранном ядре отсутствуют зависимости. Выполните отдельную ячейку: '
            + install_command
            + '. Затем перезапустите ядро и выполните notebook сверху вниз. '
            + 'Или включите INSTALL_MISSING_DEPENDENCIES в этой ячейке.'
        )
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
    importlib.invalidate_caches()
    if any(importlib.util.find_spec(module) is None for module in required_packages):
        raise RuntimeError('Установка завершена. Перезапустите ядро и выполните notebook сверху вниз.')
print('Зависимости выбранного ядра доступны.')


In [ ]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
from io import BytesIO
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import hashlib
import json
import os
import re
import subprocess

import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from pptx import Presentation
from pptx.chart.data import CategoryChartData
from pptx.dml.color import RGBColor
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.enum.shapes import MSO_SHAPE
from pptx.opc.constants import RELATIONSHIP_TYPE as RT
from pptx.opc.package import _Relationship
from pptx.opc.packuri import PackURI
from pptx.util import Inches, Pt


## Настройки
Пустая папка output допустима. Два результата создаются либо обновляются под теми же именами после проверки. RUN_CHECKS включает холодный запуск с заблокированными входными PPTX; RENDER_PREVIEWS проверяет уже готовые результаты.


In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'excel_loading.ipynb').is_file():
    raise FileNotFoundError('Запустите ядро notebook из корня проекта.')
OUTPUT_DIR = ROOT / 'output'
CENTERS = ROOT / 'Исходники/Соответствие названий центров в презентации и эксель Nephrolife.xlsx'
REPORT_MONTH = 'сентябрь'
BRANCH_COUNTS = (1, 3, 6, 11, 18)
PROCEDURE_BRANCH_COUNTS = (1, 3, 6, 11, 19)
LOAD_BRANCH_COUNTS = (1, 3, 6, 11, 18)
GENERATE_PRESENTATIONS = True
RUN_CHECKS = True
RENDER_PREVIEWS = False
LOGO = ROOT / 'resources/nephrolife_logo.jpg'
PATIENT_CENTERS = ('Атырау-Баймуханова', 'Актобе 12 мкр', 'Шымкент Абая', 'Караганда-Строителей', 'Рудный', 'Астана-Биржан Сал', 'Шымкент-Жандосова', 'Кентау', 'Караганда-Аманжолова', 'Астана-Сарыарка', 'Экибастуз', 'Костанай', 'Павлодар-Павлова', 'Петропавловск', 'Павлодар-Дюсенова', 'Лисаковск', 'Усть-Каменогорск', 'Актобе АМЦ')
PROCEDURE_CENTERS = ('Атырау-Баймуханова', 'Шымкент Абая', 'Шымкент-Жандосова', 'Астана-Биржан Сал', 'Павлодар-Павлова', 'Павлодар-Дюсенова', 'Актобе 12 мкр', 'Рудный', 'Петропавловск', 'Кентау', 'Караганда-Аманжолова', 'Караганда-Строителей', 'Экибастуз', 'Костанай', 'Астана-Сарыарка', 'Усть-Каменогорск', 'Лисаковск', 'Атырау-Нурсая', 'Актобе АМЦ')
LOAD_CENTERS = ('Астана-Биржан Сал', 'Астана-Сарыарка', 'Караганда-Строителей', 'Шымкент Абая', 'Кентау', 'Рудный', 'Шымкент-Жандосова', 'Актобе 12 мкр', 'Костанай', 'Павлодар-Дюсенова', 'Экибастуз', 'Караганда-Аманжолова', 'Павлодар-Павлова', 'Петропавловск', 'Лисаковск', 'Атырау-Баймуханова', 'Усть-Каменогорск', 'Актобе АМЦ')
DRUG_CENTERS = ('Шымкент \nАбая', 'Астана \nСарыарка', 'Кентау', 'Актобе \nАМЦ', 'Атырау \nНурсая', 'Экибастуз', 'Актобе', 'Рудный', 'Атырау\nБаймуханова', 'Лисаковск', 'Шымкент', 'УК', 'Караганда \nСтроителей', 'Астана \nБиржан Сал', 'Павлодар \nДюсенова', 'Караганда \nЦАД', 'Петропавловск', 'Костанай', 'Павлодар \nПавлова')
SLIDE5_MONTHS = ('январь','февраль','март','апрель','май','июнь','июль','август','сентябрь')
SLIDE5_ARTICLES = ('Дарбэпоэтин альфа (Аранесп),30мкг,шпр.','Дарбэпоэтин альфа (Аранесп),20мкг,шпр.')
SLIDE5_CODES = ('211.01','211.02')

NEW_DRUG_ARTICLES = ('Эритропоэтин короткого действия','Сахарат железа (Венофер)(Аджифер)')
MINERAL_DRUG_ARTICLES = ('Севеламера карбонат','Кальция ацетат (кальций ацтек)')
NEW_DRUG_CODES = ('211.03','211.04')
MINERAL_DRUG_CODES = ('211.08','211.09')
PARICALCITOL_DRUG_CODES = ('211.07','211.10')
CALDEROL_DRUG_CODES = ('211.11','211.12')
EXTRA_DRUG_PAIRS = (('new',NEW_DRUG_CODES),('mineral',MINERAL_DRUG_CODES),
                    ('paricalcitol',PARICALCITOL_DRUG_CODES),('calderol',CALDEROL_DRUG_CODES))
ZERO_ALLOWED_CODES = SLIDE5_CODES + NEW_DRUG_CODES + MINERAL_DRUG_CODES + PARICALCITOL_DRUG_CODES + CALDEROL_DRUG_CODES

SLIDE6_MONTHS = SLIDE5_MONTHS
FINANCIAL_BRANCH_COUNTS = BRANCH_COUNTS
FINANCIAL_DISPLAY_DIVISOR = Decimal('1000')  # финансовый бюджет в тенге, слайд в тыс. тенге

SLIDE7_PERIODS = (('1 квартал',SLIDE5_MONTHS[:3]),('2 квартал',SLIDE5_MONTHS[3:6]),('3 квартал',SLIDE5_MONTHS[6:9]))
QUARTER_BRANCH_COUNTS = BRANCH_COUNTS


## Исходные книги

Читаются `NL_Кол. Бюджет.xlsx` и `NL_Фин. Бюджет.xlsx` из папки `Исходники`.
Запускайте ядро из корня проекта; при другом расположении измените `DATA_DIR` на папку с этими книгами.


In [ ]:
DATA_DIR = Path.cwd() / 'Исходники'
excel_files = {
    'financial': DATA_DIR / 'NL_Фин. Бюджет.xlsx',
    'quantity': DATA_DIR / 'NL_Кол. Бюджет.xlsx',
}

missing_files = [key for key, path in excel_files.items() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Не найдены книги: ' + ', '.join(missing_files)
        + '. Проверьте DATA_DIR и имена файлов.'
    )


## Загрузка всех листов

`sheet_name=None` возвращает словарь листов. `header=None` сохраняет все строки, поскольку строка заголовков ещё не согласована. `dtype=object` отключает подбор типа для всей колонки; `keep_default_na=False` сохраняет текстовые маркеры вроде `NA`, а пустые ячейки обычно представлены пустой строкой. Пропуски не заменяются нулями.

Это первичное чтение значений, а не полная модель Excel: оформление и объединения не сохраняются в DataFrame, ошибки ячеек могут стать пропусками. Формулы не пересчитываются — читаются сохранённые Excel результаты, которые могут отсутствовать или быть устаревшими. Эти случаи требуют отдельной проверки перед расчётами.

In [ ]:
workbooks = {
    key: pd.read_excel(
        path,
        sheet_name=None,
        header=None,
        dtype=object,
        keep_default_na=False,
        engine='openpyxl',
    )
    for key, path in excel_files.items()
}

financial_sheets = workbooks['financial']
quantity_sheets = workbooks['quantity']

In [ ]:
sheet_summary = pd.DataFrame([
    {'book': key, 'sheet': sheet, 'rows': frame.shape[0], 'columns': frame.shape[1]}
    for key, sheets in workbooks.items()
    for sheet, frame in sheets.items()
])
sheet_summary

## Отдельные DataFrame филиалов двух книг NL

`quantity_df` — количественный бюджет, `financial_df` — финансовый бюджет.
Для каждой книги применяется прежняя методика: одна строка исходной статьи × филиал × период,
семь колонок: код статьи, филиал, наименование статьи, месяц, утверждено, факт по норме, факт по 1С.
В поле «Месяц» сохраняются только действительные названия 12 месяцев; сводные периоды исключены.
Книги не объединяются, показатели не пересчитываются и не округляются.

Используются те же 20 листов филиалов, строки 9–141 количественной книги и 9–193 финансовой.
Заголовки кодов (B6), статей (C6), месяцы (строка 6) и три показателя (строка 7) проверяются.
Сводные и вспомогательные листы, АУП, кварталы и годовые колонки исключены.
Итоги и повторяющиеся статьи сохраняются; пустые названия пропускаются.
Пустые показатели представлены `pd.NA`, нули сохранены, ошибки Excel восстановлены как текстовые коды.
Формулы не пересчитываются: используются сохранённые результаты Excel.


In [ ]:
BRANCH_SHEETS = [
    'Актобе', 'Актобе АМЦ', 'Астана Биржан Сал', 'Астана Сарыарка',
    'Атырау', 'Атырау Нурсая', 'Атырау Нурсая 2', 'Караганда ЦАД',
    'Караганда Строителей', 'Кентау', 'Костанай', 'Лисаковск', 'Рудный',
    'Павлодар Дюсенова', 'Павлодар Павлова', 'Экибастуз', 'Петропавловск',
    'Усть-Каменогорск (NL UK)', 'Шымкент', 'Шымкент Абая',
]
ARTICLE_ROWS = {'financial': (9, 193), 'quantity': (9, 141)}  # номера строк Excel, включительно
MONTHS = [
    'январь', 'февраль', 'март', 'апрель', 'май', 'июнь',
    'июль', 'август', 'сентябрь', 'октябрь', 'ноябрь', 'декабрь',
]
REPORT_PERIODS = tuple(MONTHS)
RESULT_COLUMNS = [
    'Код статей', 'Наименование филиала', 'Наименование статей', 'Месяц',
    'Утверждено', 'Факт по норме', 'Факт по 1С',
]


In [ ]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter


def header_text(value):
    # Нормализация применяется только к заголовкам для их проверки.
    return ' '.join(str(value).split()).casefold()


def build_branches_dataframe(source_book, *, periods=None):
    periods = tuple(REPORT_PERIODS if periods is None else periods)
    if not periods or len(set(periods)) != len(periods) or any(p not in MONTHS for p in periods):
        raise ValueError("Укажите уникальные поддерживаемые периоды.")
    if source_book not in ARTICLE_ROWS:
        raise ValueError("Источник должен быть 'financial' или 'quantity'.")
    sheets = workbooks[source_book]
    missing = [sheet for sheet in BRANCH_SHEETS if sheet not in sheets]
    if missing:
        raise ValueError('Не найдены листы филиалов: ' + ', '.join(missing))

    first_row, last_row = ARTICLE_ROWS[source_book]
    records, origins, issues = [], [], []
    # pandas превращает ошибки ячеек Excel в NaN. Второе чтение позволяет
    # восстановить коды ошибок, не смешивая их с пустыми значениями.
    cached_book = load_workbook(excel_files[source_book], read_only=True, data_only=True)
    try:
        for branch in BRANCH_SHEETS:
            frame = sheets[branch]
            if frame.shape[0] < last_row or frame.shape[1] < 56:
                raise ValueError(f'Недостаточный размер основной таблицы: {branch}')
            if header_text(frame.iat[5, 1]) != 'код статей':
                raise ValueError(f'Не найден заголовок кодов в B6: {branch}')
            if header_text(frame.iat[5, 2]) != 'наименование статей':
                raise ValueError(f'Не найден заголовок статей в C6: {branch}')

            month_columns = {}
            for column, value in enumerate(frame.iloc[5]):
                month = header_text(value)
                if month not in periods:
                    continue
                if month in month_columns:
                    raise ValueError(f'Повторяющийся заголовок месяца: {branch}, {month}')
                expected = ['утвержд.', 'факт по норме', 'факт по 1с']
                actual = [header_text(frame.iat[6, column + offset]) for offset in range(3)]
                if actual != expected:
                    raise ValueError(f'Неожиданные показатели месяца: {branch}, {month}')
                month_columns[month] = column
            if set(month_columns) != set(periods):
                raise ValueError(f'Не найдены все запрошенные периоды: {branch}')

            cached_rows = cached_book[branch].iter_rows(
                min_row=first_row, max_row=last_row, max_col=frame.shape[1],
            )
            for excel_row, cells in enumerate(cached_rows, start=first_row):
                article = frame.iat[excel_row - 1, 2]
                code_cell = cells[1]
                if code_cell.data_type == 'e':
                    raise ValueError(f'Ошибка Excel вместо кода статьи: {branch}, B{excel_row}')
                article_code = code_cell.value
                if article_code is None or (isinstance(article_code,str) and not article_code.strip()):
                    article_code = pd.NA
                if cells[2].data_type == 'e':
                    raise ValueError(f'Ошибка Excel вместо названия статьи: {branch}, C{excel_row}')
                if pd.isna(article) or article == '' or (isinstance(article, str) and not article.strip()):
                    continue
                if not isinstance(article, str):
                    raise ValueError(f'Название статьи не является текстом: {branch}, C{excel_row}')
                for month in periods:
                    column = month_columns[month]
                    values = []
                    coordinates = []
                    for offset, metric in enumerate(RESULT_COLUMNS[4:]):
                        coordinate = f'{get_column_letter(column + offset + 1)}{excel_row}'
                        coordinates.append(coordinate)
                        cell = cells[column + offset]
                        value = frame.iat[excel_row - 1, column + offset]
                        if cell.data_type == 'e':
                            value = cell.value
                            issues.append({
                                'Строка DataFrame': len(records), 'Филиал': branch,
                                'Месяц': month, 'Показатель': metric,
                                'Ячейка': coordinate, 'Ошибка Excel': value,
                            })
                        elif pd.isna(value) or value == '':
                            value = pd.NA
                        values.append(value)
                    records.append([article_code, branch, article, month, *values])
                    origins.append([source_book, branch, excel_row, *coordinates])
    finally:
        cached_book.close()

    result = pd.DataFrame(records, columns=RESULT_COLUMNS, dtype=object)
    provenance = pd.DataFrame(origins, columns=[
        'Книга', 'Лист', 'Строка Excel',
        'Ячейка Утверждено', 'Ячейка Факт по норме', 'Ячейка Факт по 1С',
    ])
    diagnostics = pd.DataFrame(issues, columns=[
        'Строка DataFrame', 'Филиал', 'Месяц', 'Показатель', 'Ячейка', 'Ошибка Excel',
    ])
    return result, provenance, diagnostics


quantity_df, quantity_provenance, quantity_diagnostics = build_branches_dataframe('quantity')
financial_df, financial_provenance, financial_diagnostics = build_branches_dataframe('financial')

# Совместимость с прежними ячейками исследования количественного бюджета.
branches_df = quantity_df
branches_provenance = quantity_provenance
branches_diagnostics = quantity_diagnostics


In [ ]:
# Только действительные месяцы; происхождение и диагностика согласованы с таблицами.
for frame, provenance, diagnostics in [
    (quantity_df, quantity_provenance, quantity_diagnostics),
    (financial_df, financial_provenance, financial_diagnostics),
]:
    assert set(frame['Месяц']) == set(MONTHS)
    assert frame.index.equals(provenance.index)
    assert diagnostics.empty or diagnostics['Месяц'].isin(MONTHS).all()

# Только размеры и диагностика, без значений показателей.
dataframe_summary = pd.DataFrame([
    {
        'Книга': key,
        'Листов': len(workbooks[key]),
        'Строк': len(frame),
        'Филиалов': frame['Наименование филиала'].nunique(),
        'Периодов': frame['Месяц'].nunique(),
        'Ошибок Excel': len(diagnostics),
    }
    for key, frame, diagnostics in [
        ('quantity', quantity_df, quantity_diagnostics),
        ('financial', financial_df, financial_diagnostics),
    ]
])
dataframe_summary


## Доступ к результатам

Основные таблицы: `quantity_df` и `financial_df`.
Координаты источников с тем же индексом: `quantity_provenance` и `financial_provenance`.
Ошибки Excel: `quantity_diagnostics` и `financial_diagnostics`.
Числовые показатели презентаций берутся только из этих двух таблиц.
Происхождение служит для идентификации исходных строк; справочник задаёт соответствия названий, оформление задаётся в коде.
Все исходные листы доступны отдельно в `quantity_sheets` и `financial_sheets`.

`branches_df`, `branches_provenance`, `branches_diagnostics` оставлены как ссылки на количественный бюджет.
Для локального просмотра можно выполнить `quantity_df.head()` или `financial_df.head()` в отдельной ячейке.
Перед сохранением удаляйте результаты с содержимым таблиц.
Не суммируйте итоговые строки вместе с детализацией без согласованного правила.
При смене версии книг перепроверьте список филиалов и границы `ARTICLE_ROWS`.


## Проверенные модели
Числовой источник — переданные DataFrame; справочник связывает явные подписи центров с листами. Периоды, строки и формулы не изменены.

Модель financial_df использует норму, факт 1С и их разность; суммы переводятся из тенге в тысячи без округления расчётов.


In [ ]:
def _center_key(text):
    key = re.sub(r'[^а-яёa-z0-9]', '', str(text).casefold().replace('\\n', ''))
    return key.replace('12мкрн', '12мкр')


def _number(value):
    if pd.isna(value) or isinstance(value, (str, bool)):
        raise ValueError('Показатель отсутствует или не является числом.')
    result = Decimal(str(value))
    if not result.is_finite() or result < 0 or result != result.to_integral_value():
        raise ValueError('Ожидается целое неотрицательное значение показателя.')
    return result


def _percent(value):
    return str((value * 100).quantize(Decimal('0.1'), rounding=ROUND_HALF_UP)).replace('.', ',') + '%'


def _integer(value):
    return f'{int(value):,}'.replace(',', ' ')


def _counted(value, forms):
    number = abs(int(value))
    if 11 <= number % 100 <= 14:
        word = forms[2]
    elif number % 10 == 1:
        word = forms[0]
    elif 2 <= number % 10 <= 4:
        word = forms[1]
    else:
        word = forms[2]
    return f'{_integer(value)} {word}'


def _count_model(quantity_df, quantity_provenance, labels, *, month, article_row, article_name, number=_number, months=None, article_code=None):
    """Read an explicit source row and complete monthly selection from the supplied DataFrame."""
    if not labels or len(set(map(_center_key,labels)))!=len(labels):
        raise ValueError('Нужен непустой список разных центров.')
    selected_months = tuple(months) if months is not None else (month,)
    if not selected_months or len(set(selected_months)) != len(selected_months) or any(m not in MONTHS for m in selected_months):
        raise ValueError("Ожидаются уникальные действительные месяцы.")
    if month not in (*MONTHS, "9 месяцев"):
        raise ValueError('Неизвестный месяц.')
    if not quantity_df.index.is_unique or not quantity_df.index.equals(quantity_provenance.index):
        raise ValueError('Индексы DataFrame и происхождения не совпадают или не уникальны.')
    if not quantity_provenance['Книга'].eq('quantity').all():
        raise ValueError('Ожидается происхождение количественного бюджета.')
    wb = load_workbook(CENTERS, read_only=True, data_only=True)
    try:
        mapping = {}
        for title, branch in list(wb.active.values)[1:]:
            if not title or not branch:
                continue
            # Explicit suffix difference between supplied directory and workbook.
            if branch == 'Усть-Каменогорск':
                branch = 'Усть-Каменогорск (NL UK)'
            key = _center_key(title)
            if key in mapping:
                raise ValueError('Повторяющееся название в справочнике центров.')
            mapping[key] = branch
    finally:
        wb.close()
    rows = []
    for label in labels:
        branch = mapping.get(_center_key(label))
        if branch is None:
            raise ValueError('Центр образца не найден в справочнике.')
        selected = quantity_df.loc[
            quantity_df['Наименование филиала'].eq(branch)
            & quantity_df['Месяц'].isin(selected_months)
            & quantity_provenance['Строка Excel'].eq(article_row)
        ]
        if (len(selected) != len(selected_months)
                or set(selected['Месяц']) != set(selected_months)
                or selected['Месяц'].duplicated().any()):
            raise ValueError('Для центра требуется ровно одна исходная строка каждого выбранного месяца.')
        plan, actual = Decimal(0), Decimal(0)
        for idx, row in selected.iterrows():
            origin = quantity_provenance.loc[idx]
            if origin['Лист'] != branch or (
                    row['Код статей'] != article_code if article_code is not None
                    else row['Наименование статей'].strip() != article_name):
                raise ValueError('Изменилась статья в выбранной строке источника.')
            # Validate every value before summing: no skipna and no replacement of missing data.
            plan += number(row['Утверждено'])
            actual += number(row['Факт по 1С'])
        if plan == 0:
            raise ValueError('Нулевой итоговый план: правило расчёта исполнения требует согласования.')
        rows.append(dict(label=label.strip(), branch=branch, plan=plan, actual=actual,
                         ratio=actual / plan, source_index=selected.index[0],
                         source_indices=tuple(selected.index)))
    if len({r['branch'] for r in rows}) != len(rows):
        raise ValueError('Один филиал сопоставлен нескольким центрам.')
    # Original is sorted by execution; stable ordering resolves equal ratios.
    return sorted(rows, key=lambda r: r['ratio'], reverse=True)

def patient_model(quantity_df, quantity_provenance, labels, *, month, use_code=False):
    return _count_model(quantity_df, quantity_provenance, labels, month=month,
                        article_row=14, article_name='Количество пациентов',
                        article_code=3 if use_code else None)


def procedure_model(quantity_df, quantity_provenance, labels, *, month='9 месяцев'):
    if month != '9 месяцев':
        raise ValueError('Для процедур подтверждена сумма января–сентября.')
    return _count_model(quantity_df, quantity_provenance, labels, month=month,
                        article_row=15, article_name='Количество процедур', number=_procedure_number,
                        months=MONTHS[:9])

def _procedure_number(value):
    if pd.isna(value) or isinstance(value, (str, bool)):
        raise ValueError('Показатель процедур отсутствует или не является числом.')
    result = Decimal(str(value))
    if not result.is_finite() or result < 0:
        raise ValueError('Ожидается конечное неотрицательное число процедур.')
    return result


def _procedure_counted(value, forms):
    if value == value.to_integral_value():
        return _counted(value, forms)
    whole, fraction = format(abs(value), 'f').split('.')
    whole = f'{int(whole):,}'.replace(',', ' ')
    return ('-' if value < 0 else '') + whole + ',' + fraction.rstrip('0') + ' процедуры'

def load_model(quantity_df, quantity_provenance, labels, *, month='сентябрь'):
    """September patients / apparatus, separately for approved plan and 1C actual."""
    if month not in MONTHS:
        raise ValueError('Для нагрузки требуется действительное название месяца.')
    patients = patient_model(quantity_df, quantity_provenance, labels, month=month, use_code=True)
    result = []
    for patient in patients:
        branch = patient['branch']
        selected = quantity_df.loc[
            quantity_df['Наименование филиала'].eq(branch)
            & quantity_df['Месяц'].eq(month)
            & quantity_provenance['Строка Excel'].eq(13)
        ]
        if len(selected) != 1:
            raise ValueError('Для центра требуется ровно одна строка аппаратов.')
        idx = selected.index[0]
        row, origin = selected.iloc[0], quantity_provenance.loc[idx]
        if origin['Лист'] != branch or row['Код статей'] != 2:
            raise ValueError('Строка 13 должна содержать количество аппаратов этого центра.')
        apparatus_plan = _number(row['Утверждено'])
        apparatus_actual = _number(row['Факт по 1С'])
        if apparatus_plan == 0 or apparatus_actual == 0:
            raise ValueError('Нулевая численность аппаратов: нагрузку вычислить нельзя.')
        result.append(dict(label=patient['label'], branch=branch,
                           patient_plan=patient['plan'], patient_actual=patient['actual'],
                           apparatus_plan=apparatus_plan, apparatus_actual=apparatus_actual,
                           plan=patient['plan'] / apparatus_plan,
                           actual=patient['actual'] / apparatus_actual,
                           patient_index=patient['source_index'], apparatus_index=idx))
    return sorted(result, key=lambda row: row['actual'], reverse=True)


def _load_text(value):
    return str(value.quantize(Decimal('0.1'), rounding=ROUND_HALF_UP)).replace('.', ',')


def slide5_center_map():
    # Короткие подписи слайда 5 отличаются от подписей слайда пациентов.
    book = load_workbook(CENTERS, read_only=True, data_only=True)
    mapping = {}
    try:
        for label, branch in list(book.active.values)[1:]:
            if label and branch:
                branch = 'Усть-Каменогорск (NL UK)' if branch == 'Усть-Каменогорск' else branch
                key = _center_key(label)
                if key in mapping:
                    raise ValueError('Неоднозначный справочник центров.')
                mapping[key] = branch
    finally:
        book.close()
    for branch in BRANCH_SHEETS:
        mapping.setdefault(_center_key(branch), branch)
    mapping[_center_key('УК')] = 'Усть-Каменогорск (NL UK)'
    return mapping


def drug_number(value):
    if pd.isna(value) or isinstance(value, (str, bool)):
        raise ValueError('Расход препарата отсутствует или содержит ошибку/текст.')
    result = Decimal(str(value))
    if not result.is_finite() or result < 0:
        raise ValueError('Расход препарата должен быть конечным неотрицательным числом.')
    return result


def slide5_value(value, branch, article_code, month, metric):
    # Только согласованные коды и точечные августовские пропуски факта 1С.
    if (pd.isna(value) and branch == 'Атырау Нурсая'
            and article_code in ZERO_ALLOWED_CODES and month == 'август' and metric == 'Факт по 1С'):
        return Decimal(0)
    return drug_number(value)


def drug_value(value,branch,article_code,month,metric,book):
    if book=='quantity':return slide5_value(value,branch,article_code,month,metric)
    # Сентябрьский финансовый факт только согласованных кодов равен нулю при пропуске.
    if (pd.isna(value) and book=='financial' and article_code in ZERO_ALLOWED_CODES
            and month=='сентябрь' and metric=='Факт по 1С'):
        return Decimal(0)
    return drug_number(value)


def drug_model(frame, provenance, labels, *, months, book, first_metric, divisor, articles=SLIDE5_CODES):
    """Select metrics by article code; keep the source name for slide labels."""
    if len(articles) != 2 or len(set(articles)) != 2:
        raise ValueError('Для слайда нужны два разных кода статей.')
    if (not months or len(set(months)) != len(months)
            or any(m not in MONTHS for m in months)):
        raise ValueError('Некорректный набор месяцев.')
    if not frame.index.is_unique or not frame.index.equals(provenance.index):
        raise ValueError('Индексы данных и происхождения не совпадают.')
    if not provenance['Книга'].eq(book).all():
        raise ValueError('Неверная книга происхождения данных препаратов.')
    if not labels or len(set(map(_center_key, labels))) != len(labels):
        raise ValueError('Нужен непустой список разных центров.')
    mapping = slide5_center_map()
    branches = [mapping.get(_center_key(label)) for label in labels]
    if any(b is None for b in branches) or len(set(branches)) != len(branches):
        raise ValueError('Центры не сопоставлены однозначно.')
    models = []
    for article_code in articles:
        rows = []
        article_name = None
        for label, branch in zip(labels, branches):
            selected = frame.loc[frame['Наименование филиала'].eq(branch)
                                 & frame['Код статей'].eq(article_code)
                                 & frame['Месяц'].isin(months)]
            origin = provenance.loc[selected.index]
            if (len(selected) != len(months)
                    or set(selected['Месяц']) != set(months)
                    or selected['Месяц'].duplicated().any()
                    or origin['Строка Excel'].nunique() != 1
                    or not origin['Лист'].eq(branch).all()):
                raise ValueError('Для кода статьи нужны однозначные строки всех выбранных месяцев.')
            names = selected['Наименование статей'].unique()
            if (len(names) != 1 or not isinstance(names[0],str) or not names[0].strip()
                    or (article_name is not None and article_name != names[0])):
                raise ValueError('Код статьи имеет неоднозначное наименование в книге.')
            article_name = names[0]
            totals = {}
            for metric in dict.fromkeys((first_metric, 'Факт по норме', 'Факт по 1С')):
                totals[metric] = sum((drug_value(r[metric], branch, article_code, r['Месяц'], metric,book)
                                      for _,r in selected.iterrows()), Decimal(0))/divisor
            plan, actual = totals[first_metric], totals['Факт по 1С']
            rows.append(dict(label=label, branch=branch, article=article_name, article_code=article_code,
                             plan=plan, actual=actual, norm=totals['Факт по норме'],
                             difference=totals['Факт по норме']-actual, source_indices=tuple(selected.index)))
        models.append(rows)
    return models


def slide5_model(frame,provenance,labels,*,months=SLIDE5_MONTHS,articles=SLIDE5_CODES):
    return drug_model(frame,provenance,labels,months=months,book='quantity',first_metric='Утверждено',divisor=Decimal(1),articles=articles)


def slide6_model(frame,provenance,labels,*,months=SLIDE6_MONTHS,articles=SLIDE5_CODES):
    return drug_model(frame,provenance,labels,months=months,book='financial',first_metric='Факт по норме',
                      divisor=FINANCIAL_DISPLAY_DIVISOR,articles=articles)


def slide7_model(frame,provenance,labels,*,articles=SLIDE5_CODES):
    quarterly=[drug_model(frame,provenance,labels,months=months,book='quantity',
                          first_metric='Факт по норме',divisor=Decimal(1),articles=articles) for _,months in SLIDE7_PERIODS]
    result=[]
    for article_index in range(len(articles)):
        periods=[]
        for (period,months),models in zip(SLIDE7_PERIODS,quarterly):
            rows=models[article_index]
            norm=sum((r['norm'] for r in rows),Decimal(0));actual=sum((r['actual'] for r in rows),Decimal(0))
            periods.append(dict(label=period,article=rows[0]['article'],article_code=articles[article_index],
                                 plan=norm,norm=norm,actual=actual,difference=norm-actual,
                                months=tuple(months),centers=tuple(labels)))
        norm=sum((r['norm'] for r in periods),Decimal(0));actual=sum((r['actual'] for r in periods),Decimal(0))
        periods.append(dict(label='9 месяцев',article=periods[0]['article'],article_code=articles[article_index],
                             plan=norm,norm=norm,actual=actual,difference=norm-actual,
                            months=SLIDE5_MONTHS,centers=tuple(labels)))
        result.append(periods)
    return result


## Программное оформление
Макеты, диаграммы и текст создаются с нуля, без чтения внешнего PPTX.


In [ ]:
from pptx.enum.chart import (XL_CHART_TYPE, XL_DATA_LABEL_POSITION, XL_TICK_MARK, XL_TICK_LABEL_POSITION)
from pptx.oxml.xmlchemy import OxmlElement
STYLE = {'font':'Arial Narrow','teal':'1BAA98','gray':'BFBFBF','blue':'4472C4','red':'FF0000','green':'008000'}


def new_presentation():
    # Создаём стандартную пустую презентацию; пользовательский PPTX не открывается.
    deck = Presentation()
    deck.slide_width,deck.slide_height = Inches(13+1/3),Inches(7.5)
    deck.core_properties.author,deck.core_properties.title = 'Papagun','Отчёт NL'
    return deck


def presentation_bytes(deck):
    stream = BytesIO();deck.save(stream)
    return stream.getvalue()


def text_box(slide,text,left,top,width,height,*,size=14,color='182230',bold=False,fill=None,name=None,align=PP_ALIGN.LEFT):
    shape = slide.shapes.add_textbox(Inches(left),Inches(top),Inches(width),Inches(height))
    if name:shape.name = name
    if fill:
        shape.fill.solid();shape.fill.fore_color.rgb = RGBColor.from_string(fill)
    frame = shape.text_frame;frame.word_wrap = False
    frame.margin_top = frame.margin_bottom = 0
    frame.margin_left = frame.margin_right = Inches(.04)
    paragraph = frame.paragraphs[0];paragraph.alignment = align
    run = paragraph.add_run();run.text = text
    run.font.name,run.font.size,run.font.bold = STYLE['font'],Pt(size),bold
    run.font.color.rgb = RGBColor.from_string(color)
    return shape


def add_header(slide,title,*,width=6.4):
    text_box(slide,title,0,.15,width,.36,size=18,bold=True,fill=STYLE['teal'])
    if LOGO.is_file():
        slide.shapes.add_picture(str(LOGO),Inches(12.74),Inches(.04),width=Inches(.53))


def add_summary(slide,lines,*,left=6.55,top=.04,width=3.7,height=1.20):
    shape = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE,Inches(left),Inches(top),Inches(width),Inches(height))
    shape.name = 'Сводка';shape.fill.solid();shape.fill.fore_color.rgb = RGBColor.from_string(STYLE['gray'])
    shape.line.fill.background();frame = shape.text_frame;frame.word_wrap = False
    frame.margin_left = frame.margin_right = Inches(.12)
    frame.margin_top = frame.margin_bottom = Inches(.04)
    for i,(text,color) in enumerate(lines):
        paragraph = frame.paragraphs[0] if i==0 else frame.add_paragraph()
        paragraph.space_before = paragraph.space_after = Pt(0)
        paragraph.alignment = PP_ALIGN.LEFT
        run = paragraph.add_run();run.text = text
        run.font.name,run.font.size = STYLE['font'],Pt(13)
        run.font.color.rgb = RGBColor.from_string(color)
    return shape


def set_plot_layout(chart,*,x,y,w,h):
    area = chart._chartSpace.xpath('.//c:plotArea')[0]
    for node in area.xpath('./c:layout'):area.remove(node)
    layout = OxmlElement('c:layout');manual = OxmlElement('c:manualLayout')
    for tag,val in [('layoutTarget','inner'),('xMode','edge'),('yMode','edge'),('x',x),('y',y),('w',w),('h',h)]:
        node = OxmlElement('c:'+tag);node.set('val',str(val));manual.append(node)
    layout.append(manual);area.insert(0,layout)


def numeric_chart(slide,labels,series,left,top,width,height,*,fmt='0',compact_base=None,drug=False,total=False,financial=False):
    if not labels:raise ValueError('Диаграмме нужен хотя бы один центр.')
    data = CategoryChartData(number_format=fmt)
    data.categories = [str(label).replace('\\n','\n') for label in labels]
    for name,values,color in series:data.add_series(name,[float(v) for v in values])
    shape = slide.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED,Inches(left),Inches(top),Inches(width),Inches(height),data)
    chart = shape.chart;chart.has_title = chart.has_legend = False
    chart.value_axis.visible = False;chart.value_axis.has_major_gridlines = False
    chart.category_axis.tick_label_position = XL_TICK_LABEL_POSITION.LOW
    chart.category_axis.tick_labels.font.name = STYLE['font']
    chart.category_axis.tick_labels.font.size = Pt(9 if drug else 10)
    chart.category_axis.tick_labels.font.bold = True
    chart.category_axis.major_tick_mark = chart.category_axis.minor_tick_mark = XL_TICK_MARK.NONE
    chart.category_axis.format.line.color.rgb = RGBColor(0,0,0)
    txpr = chart.category_axis._element.get_or_add_txPr()
    txpr.xpath('./a:bodyPr')[0].set('rot','0' if total else '-5400000')
    plot = chart.plots[0];plot.gap_width = 70;plot.overlap = -10;plot.has_data_labels = True
    for item,(_,values,color) in zip(chart.series,series):
        item.format.fill.solid();item.format.fill.fore_color.rgb = RGBColor.from_string(color)
        item.format.line.color.rgb = RGBColor(0,0,0)
        item.invert_if_negative = False
        labels_style = item.data_labels
        labels_style.position = XL_DATA_LABEL_POSITION.OUTSIDE_END
        labels_style.show_value = True
        labels_style.show_category_name = labels_style.show_series_name = False
        labels_style.number_format,labels_style.number_format_is_linked = fmt,False
        labels_style.font.name,labels_style.font.bold = STYLE['font'],True
        labels_style.font.size = Pt(10 if total else (8 if drug else 11))
        labels_style.font.color.rgb = RGBColor.from_string(STYLE['red'] if item.name=='отк:' else '000000')
        for body in item._element.xpath('./c:dLbls/c:txPr/a:bodyPr'):
            body.set('wrap','none');body.set('rot','0' if total or item.name=='отк:' else '-5400000')
            body.set('lIns','0');body.set('rIns','0')
        if drug and item.name=='отк:':
            for point,value in zip(item.points,values):
                value=Decimal(str(value))
                point.format.fill.solid()
                point.format.fill.fore_color.rgb = RGBColor.from_string(STYLE['red'] if value<0 else STYLE['gray'])
                # PowerPoint defaults individual negative points to inverted fill unless explicitly disabled.
                dpt=item._element.get_or_add_dPt_for_point(point._idx)
                invert=OxmlElement('c:invertIfNegative');invert.set('val','0');dpt.insert(1,invert)
                label=point.data_label
                label.font.name,label.font.size,label.font.bold = STYLE['font'],Pt(10 if total else 8),True
                label.font.color.rgb = RGBColor.from_string(STYLE['red'] if value<0 else (STYLE['green'] if value>0 else '000000'))
                for body in label._element.xpath('./c:txPr/a:bodyPr'):
                    body.set('rot','0');body.set('wrap','none')
    base = compact_base if compact_base is not None else len(labels)
    fraction = (.98 if drug else .9666091006930311)*min(1,len(labels)/base)
    set_plot_layout(chart,x=(1-fraction)/2,y=.09 if drug else .06,w=fraction,h=.67 if drug else .72)
    for node in chart._chartSpace.xpath('.//c:catAx/c:lblAlgn'):node.set('val','ctr')
    values = [Decimal(str(v)) for _,items,_ in series for v in items]
    maximum,minimum = max(values),min(Decimal(0),min(values))
    chart.value_axis.maximum_scale = float(maximum*Decimal('1.45' if drug else '1.25')) if maximum else 1
    floor=minimum*Decimal(3)
    if financial and total and minimum<0:floor=min(floor,-maximum*Decimal('.3'))
    chart.value_axis.minimum_scale = float(floor)
    if drug:
        frame = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE,shape.left,shape.top,shape.width,shape.height)
        frame.fill.background();frame.line.color.rgb = RGBColor(0,0,0)
    return shape


def write_single(payload,output):
    if output is None:return payload
    path = Path(output)
    if path.suffix.lower()!='.pptx' or path.resolve().is_relative_to((ROOT/'Исходники').resolve()):
        raise ValueError('Укажите выходной PPTX вне каталога исходников.')
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('xb') as target:target.write(payload)
    return path


def _build_count_slide(df,prov,*,month,labels,kind,output=None):
    patients = kind=='patients'
    model = (patient_model if patients else procedure_model)(df,prov,labels,month=month)
    deck = new_presentation();slide = deck.slides.add_slide(deck.slide_layouts[6])
    add_header(slide,f'ПАЦИЕНТЫ: {month} 2026 г.' if patients else 'ПРОЦЕДУРЫ: январь–сентябрь 2026 г.')
    plan = sum((r['plan'] for r in model),Decimal(0));actual = sum((r['actual'] for r in model),Decimal(0))
    counted = _counted if patients else _procedure_counted
    forms = ('пациент','пациента','пациентов') if patients else ('процедура','процедуры','процедур')
    warning = STYLE['red'] if actual<plan else STYLE['blue']
    add_summary(slide,[
        (f'{month.capitalize()} 2026: {_counted(len(model),("центр","центра","центров"))} образца','000000'),
        ('План: '+counted(plan,forms),'000000'),('Факт: '+counted(actual,forms),'000000'),
        ('Отклонение: '+counted(actual-plan,forms),warning),
        (('Исполнение ' if patients else 'Исполнение: ')+_percent(actual/plan),warning)])
    shape = numeric_chart(slide,[r['label'] for r in model],[('План',[r['plan'] for r in model],STYLE['teal']),
                          ('Факт',[r['actual'] for r in model],STYLE['gray'])],
                          0,1.85,13+1/3,5.5,fmt='0' if patients else 'General',compact_base=18 if patients else 19)
    fraction = .9666091006930311*min(1,len(model)/(18 if patients else 19))
    plot_left = shape.left+shape.width*(1-fraction)/2
    for i,row in enumerate(model):
        center = plot_left+shape.width*fraction*(i+.5)/len(model)
        box = text_box(slide,_percent(row['ratio']),(center-Inches(.28))/914400,1.47,.56,.24,size=12,bold=True,
                       color=STYLE['red'] if row['ratio']<Decimal('.95') else STYLE['blue'],name=f'Исполнение {i+1}',align=PP_ALIGN.CENTER)
        box.fill.solid();box.fill.fore_color.rgb = RGBColor(255,255,255)
        box.line.color.rgb = RGBColor(0,0,0);box.line.width = Pt(.8)
        box.text_frame.margin_left = box.text_frame.margin_right = 0
    return write_single(presentation_bytes(deck),output)


def build_slide2(df,prov,*,month='сентябрь',output=None,labels=None):
    return _build_count_slide(df,prov,month=month,kind='patients',labels=PATIENT_CENTERS if labels is None else labels,output=output)


def build_slide3(df,prov,*,output=None,labels=None):
    return _build_count_slide(df,prov,month='9 месяцев',kind='procedures',labels=PROCEDURE_CENTERS if labels is None else labels,output=output)


def build_slide4(df,prov,*,month='сентябрь',output=None,labels=None):
    model = load_model(df,prov,LOAD_CENTERS if labels is None else labels,month=month)
    deck = new_presentation();slide = deck.slides.add_slide(deck.slide_layouts[6])
    add_header(slide,f'НАГРУЗКА НА АППАРАТЫ: {month} 2026 г.',width=8)
    numeric_chart(slide,[r['label'] for r in model],[('Столбец1',[r['actual'] for r in model],STYLE['teal'])],
                  0,1.45,13+1/3,4.7,fmt='0.0',compact_base=18)
    sums = {k:sum((r[k] for r in model),Decimal(0)) for k in ('patient_plan','patient_actual','apparatus_plan','apparatus_actual')}
    add_summary(slide,[('','000000'),('Всего аппаратов  '+_integer(sums['apparatus_actual']),'000000'),
        ('Плановая нагрузка '+_load_text(sums['patient_plan']/sums['apparatus_plan']),'000000'),
        ('Фактическая нагрузка на аппараты '+_load_text(sums['patient_actual']/sums['apparatus_actual']),'000000')],
        left=.3,top=6.30,width=4.7,height=1.12)
    return write_single(presentation_bytes(deck),output)


def slide5_labels():return list(DRUG_CENTERS)


def drug_execution_text(norm,actual):
    if actual==0:return 'н/д'
    return str((norm*100/actual).quantize(Decimal('1'),rounding=ROUND_HALF_UP))+'%'


def render_drug_models(models,*,heading,series_names,financial=False,articles=SLIDE5_CODES):
    deck = new_presentation();slide = deck.slides.add_slide(deck.slide_layouts[6])
    add_header(slide,heading,width=11.8)
    for i,(article_code,rows) in enumerate(zip(articles,models)):
        article=rows[0]['article']
        if rows[0]['article_code']!=article_code:raise ValueError('Код статьи и модель слайда не совпадают.')
        top,height = (.72,2.88) if i==0 else (3.82,3.15)
        totals = {k:sum((r[k] for r in rows),Decimal(0)) for k in ('plan','actual','difference')}
        for total,left,width,categories,points in [(False,.15,10.63,[r['label'] for r in rows],rows),
                (True,10.82,2.32,[f'Итого по показанным\nцентрам ({len(rows)})'],[totals])]:
            series = [(name,[r[key] for r in points],color) for name,key,color in
                      zip(series_names,('plan','actual','difference'),(STYLE['teal'],STYLE['gray'],STYLE['gray']))]
            numeric_chart(slide,categories,series,left,top,width,height,fmt='#,##0',
                          compact_base=19 if not total else 1,drug=True,total=total,financial=financial)
        if financial:
            panel=text_box(slide,drug_execution_text(totals['plan'],totals['actual']),
                           12.455,top+.10,.552,.264,size=10,bold=True,color='000000',fill=STYLE['gray'],
                           name='drug_execution:'+article,align=PP_ALIGN.CENTER)
            panel.line.color.rgb=RGBColor(0,0,0)
            panel.text_frame.vertical_anchor=MSO_ANCHOR.MIDDLE
        title = text_box(slide,article,.25,top+.025,10.43,.22,size=9,bold=True,color='0070C0',
                         name='drug_article',align=PP_ALIGN.CENTER)
    return presentation_bytes(deck)


def build_slide5(frame,provenance,labels=None,*,tag='main',articles=SLIDE5_CODES):
    models=slide5_model(frame,provenance,list(DRUG_CENTERS) if labels is None else list(labels),articles=articles)
    return render_drug_models(models,heading='РАСХОДЫ ЛС: январь–сентябрь 2026 г. (количество)',
                              series_names=('План','Факт','отк:'),articles=articles),models


def build_slide6(frame,provenance,labels=None,*,tag='main',articles=SLIDE5_CODES):
    models=slide6_model(frame,provenance,list(DRUG_CENTERS) if labels is None else list(labels),articles=articles)
    return render_drug_models(models,heading='РАСХОДЫ ЛС: январь–сентябрь 2026 г. (тыс. тенге)',
                              series_names=('Факт по норме','Факт по 1С','отк:'),financial=True,articles=articles),models


def quarter_execution_info(norm,actual):
    if norm==0:return 'н/д','000000'
    text=str((actual*100/norm).quantize(Decimal(1),rounding=ROUND_HALF_UP))+'%'
    return text,STYLE['red'] if actual>norm else '000000'


def build_slide7(frame,provenance,labels=None,*,tag='main',articles=SLIDE5_CODES):
    labels=list(DRUG_CENTERS) if labels is None else list(labels)
    models=slide7_model(frame,provenance,labels,articles=articles)
    deck=new_presentation();slide=deck.slides.add_slide(deck.slide_layouts[6])
    add_header(slide,'РАСХОДЫ ЛС: по кварталам 2026 г. (количество)',width=11.8)
    for index,(article_code,rows) in enumerate(zip(articles,models)):
        article=rows[0]['article']
        if rows[0]['article_code']!=article_code:raise ValueError('Код статьи и квартальная модель не совпадают.')
        top,height=(.65,3.10) if index==0 else (3.88,3.12)
        series=[(name,[r[key] for r in rows],color) for name,key,color in
                zip(('Факт по норме','Факт по 1С','отк:'),('norm','actual','difference'),
                    (STYLE['teal'],STYLE['gray'],STYLE['gray']))]
        numeric_chart(slide,[r['label'] for r in rows],series,.15,top,12.99,height,
                      fmt='#,##0',compact_base=4,drug=True,total=True)
        text_box(slide,article,.25,top+.025,12.79,.22,size=9,bold=True,color='0070C0',
                 name='quarter_article:'+article,align=PP_ALIGN.LEFT)
        for j,row in enumerate(rows):
            center=.15+12.99*(.01+(j+.5)*.98/4)
            percent,color=quarter_execution_info(row['norm'],row['actual'])
            panel=text_box(slide,percent,center-.325,top+(.20 if j==0 else .06),.65,.24,
                           size=10,bold=True,color=color,fill=STYLE['gray'],
                           name='quarter_execution:'+article+':'+row['label'],align=PP_ALIGN.CENTER)
            panel.line.color.rgb=RGBColor(0,0,0);panel.text_frame.vertical_anchor=MSO_ANCHOR.MIDDLE
    return presentation_bytes(deck),models


## Проверки и сохранение
Проверяется содержимое каждого слайда и встроенной XLSX. Предыдущие результаты не нужны.


In [ ]:
def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def require(condition, message):
    if not condition:
        raise AssertionError(message)


def percentage(value):
    return str((value * 100).quantize(Decimal('.1'), rounding=ROUND_HALF_UP)).replace('.', ',') + '%'


def integer(value):
    return f'{int(value):,}'.replace(',', ' ')


def join_slides(paths):
    """Copy already generated OPC parts without changing slide XML or drawing."""
    if not paths or any(not isinstance(path,BytesIO) for path in paths):
        raise ValueError('Сборка принимает только созданные в памяти пакеты, без входных PPTX.')
    result = Presentation(paths[0])
    package = result.part.package
    layout = result.slides[0].slide_layout.part
    reserved = {str(p.partname) for p in package.iter_parts()}
    for path in paths[1:]:
        source = Presentation(path)
        copied = {}

        def copy_part(part):
            if part in copied:
                return copied[part]
            pattern = re.sub(r'\d+(?=\.[^.]+$)', '%d', str(part.partname))
            if '%d' not in pattern:
                stem, suffix = str(part.partname).rsplit('.', 1)
                pattern = stem + '%d.' + suffix
            index = 1
            while pattern % index in reserved:
                index += 1
            name = PackURI(pattern % index)
            reserved.add(str(name))
            clone = type(part).load(name, part.content_type, package, part.blob)
            copied[part] = clone
            for rel in part.rels.values():
                if rel.is_external:
                    target, mode = rel.target_ref, 'External'
                else:
                    target = layout if rel.reltype == RT.SLIDE_LAYOUT else copy_part(rel.target_part)
                    mode = 'Internal'
                # Preserve rIds, so slide/chart XML remains byte-for-byte intact.
                clone.rels._rels[rel.rId] = _Relationship(name.baseURI, rel.rId, rel.reltype, mode, target)
            return clone

        slide_part = copy_part(source.slides[0].part)
        rid = result.part.relate_to(slide_part, RT.SLIDE)
        result.slides._sldIdLst.add_sldId(rid)
    return result


def check_slide(slide, expected, width, height, *, kind='patients'):
    charts = [s.chart for s in slide.shapes if s.has_chart]
    require(len(charts) == 1, 'Expected one chart.')
    chart = charts[0]
    require(len(chart.series) == 2, 'Expected plan and actual series.')
    require([c.label for c in chart.plots[0].categories] == [r['label'] for r in expected], 'Wrong categories/order.')
    for series, field in zip(chart.series, ('plan', 'actual')):
        require(list(series.values) == [float(r[field]) for r in expected], 'Wrong chart values.')
    workbook = load_workbook(BytesIO(chart.part.chart_workbook.xlsx_part.blob), read_only=True, data_only=True)
    try:
        entries = list(workbook.active.values)
        require(len(entries) == len(expected) + 1, 'Stale rows in embedded workbook.')
        for entry, row in zip(entries[1:], expected):
            require(entry[:3] == (row['label'], float(row['plan']), float(row['actual'])), 'Wrong embedded values.')
    finally:
        workbook.close()
    plan = sum((r['plan'] for r in expected), Decimal(0))
    actual = sum((r['actual'] for r in expected), Decimal(0))
    summary = next(s for s in slide.shapes if s.has_text_frame and 'образца' in s.text)
    paragraphs = summary.text_frame.paragraphs
    count_match = re.search(r'(\d+)\s+центр(?:а|ов)?\s+образца', paragraphs[0].text)
    require(count_match is not None and int(count_match[1]) == len(expected), 'Wrong center count.')
    forms = ('пациент', 'пациента', 'пациентов') if kind == 'patients' else ('процедура', 'процедуры', 'процедур')
    counted = _counted if kind == 'patients' else _procedure_counted
    for paragraph, prefix, value in zip(paragraphs[1:4], ('План: ', 'Факт: ', 'Отклонение: '), (plan, actual, actual - plan)):
        require(paragraph.text == prefix + counted(value, forms), 'Wrong plan/actual/difference total.')
    execution_prefix = 'Исполнение ' if kind == 'patients' else 'Исполнение: '
    require(paragraphs[4].text == execution_prefix + percentage(actual / plan), 'Wrong total execution.')
    badges = sorted((s for s in slide.shapes if s.name.startswith('Исполнение ')), key=lambda s: s.left)
    require(len(badges) == len(expected), 'Wrong badge count.')
    for badge, row in zip(badges, expected):
        require(badge.text == percentage(row['ratio']), 'Wrong execution badge.')
    require(all(a.left + a.width <= b.left for a, b in zip(badges, badges[1:])), 'Execution badges overlap.')
    chart_shape = next(s for s in slide.shapes if s.has_chart)
    # Regression guard: physical chart width per center stays below one inch,
    # and the reduced chart is centered rather than stretched across the slide.
    layout = chart._chartSpace.xpath('.//c:plotArea/c:layout/c:manualLayout')[0]
    plot_width = chart_shape.width * float(layout.xpath('./c:w')[0].get('val'))
    plot_left = chart_shape.left + chart_shape.width * float(layout.xpath('./c:x')[0].get('val'))
    require(plot_width / len(expected) <= 914400, 'Small selection stretches columns.')
    require(abs(plot_left + plot_width / 2 - width / 2) <= 2, 'Compact plot is not centered.')
    require(chart._chartSpace.xpath('.//c:catAx/c:lblAlgn')[0].get('val') == 'ctr', 'Category labels are not centered.')
    for shape in slide.shapes:
        require(shape.left >= 0 and shape.top >= 0 and shape.left + shape.width <= width
                and shape.top + shape.height <= height, 'Shape extends beyond slide bounds.')


def check_load_slide(slide, expected, width, height):
    charts = [s.chart for s in slide.shapes if s.has_chart]
    require(len(charts) == 1, 'Expected one load chart.')
    chart = charts[0]
    require(len(chart.series) == 1 and chart.series[0].name == 'Столбец1', 'Wrong load series.')
    require([c.label for c in chart.plots[0].categories] == [r['label'] for r in expected], 'Wrong load categories.')
    require(list(chart.series[0].values) == [float(r['actual']) for r in expected], 'Wrong actual load.')
    workbook = load_workbook(BytesIO(chart.part.chart_workbook.xlsx_part.blob), read_only=True, data_only=True)
    try:
        entries = list(workbook.active.values)
        require(len(entries) == len(expected) + 1, 'Stale rows in load workbook.')
        for entry, row in zip(entries[1:], expected):
            require(entry[0] == row['label'] and abs(float(entry[1]) - float(row['actual'])) < 1e-10,
                    'Wrong embedded load.')
    finally:
        workbook.close()
    total_patients_plan = sum((r['patient_plan'] for r in expected), Decimal(0))
    total_patients_actual = sum((r['patient_actual'] for r in expected), Decimal(0))
    total_apparatus_plan = sum((r['apparatus_plan'] for r in expected), Decimal(0))
    total_apparatus_actual = sum((r['apparatus_actual'] for r in expected), Decimal(0))
    summary = next(s for s in slide.shapes if s.has_text_frame and 'Всего аппаратов' in s.text)
    paragraphs = summary.text_frame.paragraphs
    require(paragraphs[1].text.strip() == 'Всего аппаратов  ' + _integer(total_apparatus_actual),
            'Wrong total apparatus.')
    require(paragraphs[2].text == 'Плановая нагрузка ' + _load_text(total_patients_plan / total_apparatus_plan),
            'Wrong weighted planned load.')
    require(paragraphs[3].text == 'Фактическая нагрузка на аппараты ' + _load_text(total_patients_actual / total_apparatus_actual),
            'Wrong weighted actual load.')
    chart_shape = next(s for s in slide.shapes if s.has_chart)
    layout = chart._chartSpace.xpath('.//c:plotArea/c:layout/c:manualLayout')[0]
    plot_width = chart_shape.width * float(layout.xpath('./c:w')[0].get('val'))
    plot_left = chart_shape.left + chart_shape.width * float(layout.xpath('./c:x')[0].get('val'))
    require(plot_width / len(expected) <= 914400, 'Small load selection stretches columns.')
    require(abs(plot_left + plot_width / 2 - width / 2) <= 2, 'Load plot is not centered.')
    require(not chart._chartSpace.xpath('.//c:ser/c:dLbls/c:dLbl'), 'Stale source load labels remain.')
    for shape in slide.shapes:
        require(shape.left >= 0 and shape.top >= 0 and shape.left + shape.width <= width
                and shape.top + shape.height <= height, 'Load shape extends beyond slide bounds.')

def digest(path):return hashlib.sha256(Path(path).read_bytes()).digest()


def must_fail(exc,action):
    try:action()
    except exc:return
    raise AssertionError('Ожидалась ошибка '+exc.__name__)


def combine_created_slides(payloads):
    # Внутренние пакеты только из текущего процесса; внешние презентации не читаются.
    return presentation_bytes(join_slides([BytesIO(p) for p in payloads]))


def check_drug_slide(slide,models,width,height,*,financial=False,articles=SLIDE5_CODES):
    expected_names=['Факт по норме','Факт по 1С','отк:'] if financial else ['План','Факт','отк:']
    heading='РАСХОДЫ ЛС: январь–сентябрь 2026 г. ('+('тыс. тенге' if financial else 'количество')+')'
    require(any(sh.has_text_frame and sh.text==heading for sh in slide.shapes),'Неверный заголовок слайда препаратов.')
    charts = [s for s in slide.shapes if s.has_chart]
    require(len(charts)==4,'Нужно четыре диаграммы препаратов.')
    left = sorted((s for s in charts if s.left<Inches(1)),key=lambda s:s.top)
    right = sorted((s for s in charts if s.left>Inches(10)),key=lambda s:s.top)
    require(len(left)==len(right)==2,'Нужны две пары диаграмм препаратов.')
    require(left[0].height>=Inches(2.8) and left[1].height>=Inches(3.1),
            'Диаграммы препаратов слишком низкие.')
    require(left[0].top+left[0].height<left[1].top and
            right[0].top+right[0].height<right[1].top,
            'Верхние и нижние диаграммы пересекаются.')
    for article_code,rows,branch,total in zip(articles,models,left,right):
        article=rows[0]['article']
        require(all(r['article_code']==article_code and r['article']==article for r in rows),
                'Неверный код или наименование модели препарата.')
        titles = [sh for sh in slide.shapes if sh.has_text_frame and sh.text==article]
        require(len(titles)==1,'Неверная подпись препарата.')
        title=titles[0]
        require(title.left>=branch.left and title.top>=branch.top and title.left+title.width<=branch.left+branch.width
                and title.top+title.height<=branch.top+branch.height,'Подпись статьи вне рамки диаграммы.')
        require(title.text_frame.paragraphs[0].alignment==PP_ALIGN.CENTER
                and all(run.font.size and run.font.size.pt==9 for para in title.text_frame.paragraphs for run in para.runs),
                'Неверный размер или выравнивание статьи.')
        require(abs((title.left+title.width/2)-(branch.left+branch.width/2))<Inches(.02),'Статья не по центру графика.')
        expected_total = {key:sum((r[key] for r in rows),Decimal(0)) for key in ('plan','actual','difference')}
        if financial:
            panels=[sh for sh in slide.shapes if sh.name=='drug_execution:'+article]
            require(len(panels)==1,'Нет процентной панели финансового итога.')
            panel=panels[0]
            norm=sum((r['norm'] for r in rows),Decimal(0));actual=sum((r['actual'] for r in rows),Decimal(0))
            expected='н/д' if actual==0 else str((norm/actual*100).quantize(Decimal('1'),rounding=ROUND_HALF_UP))+'%'
            require(panel.text==expected,'Процент должен быть отношением суммы нормы к сумме факта 1С.')
            require(panel.left>=total.left and panel.top>=total.top and panel.left+panel.width<=total.left+total.width
                    and panel.top+panel.height<=total.top+total.height,'Процентная панель вне сводного графика.')
        for shape,points in ((branch,rows),(total,[expected_total])):
            chart = shape.chart
            require([s.name for s in chart.series]==expected_names,'Неверные серии препаратов.')
            if shape is branch:
                require([c.label for c in chart.plots[0].categories]==[r['label'].replace('\\n','\n') for r in rows],
                        'Неверные центры препаратов.')
            for series,key in zip(chart.series,('plan','actual','difference')):
                expected=[float(r[key]) for r in points]
                require(list(series.values)==expected,'Неверные данные препаратов.')
                require(series.data_labels.number_format=='#,##0','Неверная точность отображения.')
                if key=='difference':
                    for point,value in zip(series.points,expected):
                        invert=series._element.get_or_add_dPt_for_point(point._idx).xpath('./c:invertIfNegative')
                        require(len(invert)==1 and invert[0].get('val')=='0','PowerPoint может инвертировать заливку точки.')
                        require(str(point.format.fill.fore_color.rgb).upper()==(STYLE['red'] if value<0 else STYLE['gray']),
                                'Неверный цвет столбца отклонения.')
                        expected_color=STYLE['red'] if value<0 else (STYLE['green'] if value>0 else '000000')
                        require(str(point.data_label.font.color.rgb).upper()==expected_color,'Неверный цвет метки отклонения.')
            workbook = load_workbook(BytesIO(chart.part.chart_workbook.xlsx_part.blob),read_only=True,data_only=True)
            try:
                entries = list(workbook.active.values)
                require(len(entries)==len(points)+1,'Лишние строки XLSX препаратов.')
                for entry,row in zip(entries[1:],points):
                    for value,key in zip(entry[1:],('plan','actual','difference')):
                        require(abs(float(value)-float(row[key]))<1e-8,'Неверная XLSX препаратов.')
            finally:workbook.close()
        layout = branch.chart._chartSpace.xpath('.//c:plotArea/c:layout/c:manualLayout')[0]
        w = float(layout.xpath('./c:w')[0].get('val'));x = float(layout.xpath('./c:x')[0].get('val'))
        require(float(layout.xpath('./c:h')[0].get('val'))>=.66,
                'Внутренняя область диаграммы препаратов слишком низкая.')
        require(abs(x+w/2-.5)<1e-8 and branch.width*w/len(rows)<=Inches(.6),'Растянута малая выборка.')
    for shape in slide.shapes:
        require(0<=shape.left and 0<=shape.top and shape.left+shape.width<=width
                and shape.top+shape.height<=height,'Фигура выходит за слайд.')



def verify_slide5(payload,models):
    deck = Presentation(BytesIO(payload))
    require(len(deck.slides)==1,'Ожидался один слайд препаратов.')
    check_drug_slide(deck.slides[0],models,deck.slide_width,deck.slide_height)


def build_payloads(df,prov,fdf,fprov,*,month=REPORT_MONTH):
    main_payloads = [build_slide2(df,prov,month=month),build_slide3(df,prov),build_slide4(df,prov,month=month)]
    main_models = [('patients',patient_model(df,prov,PATIENT_CENTERS,month=month)),
                   ('procedures',procedure_model(df,prov,PROCEDURE_CENTERS)),
                   ('load',load_model(df,prov,LOAD_CENTERS,month=month))]
    for kind,builder,frame,origin in (('drugs',build_slide5,df,prov),
                                      ('financial_drugs',build_slide6,fdf,fprov),
                                      ('quarter_drugs',build_slide7,df,prov)):
        payload,model=builder(frame,origin)
        main_payloads.append(payload);main_models.append((kind,model))
    for prefix,articles in EXTRA_DRUG_PAIRS:
        for kind,builder,frame,origin in (('drugs',build_slide5,df,prov),
                                          ('financial_drugs',build_slide6,fdf,fprov),
                                          ('quarter_drugs',build_slide7,df,prov)):
            payload,model=builder(frame,origin,articles=articles)
            main_payloads.append(payload);main_models.append((prefix+'_'+kind,model))
    specs = [('patients',PATIENT_CENTERS,BRANCH_COUNTS),('procedures',PROCEDURE_CENTERS,PROCEDURE_BRANCH_COUNTS),
             ('load',LOAD_CENTERS,LOAD_BRANCH_COUNTS),('drugs',DRUG_CENTERS,BRANCH_COUNTS),
             ('financial_drugs',DRUG_CENTERS,FINANCIAL_BRANCH_COUNTS),('quarter_drugs',DRUG_CENTERS,QUARTER_BRANCH_COUNTS)]
    for prefix,_ in EXTRA_DRUG_PAIRS:
        specs.extend(((prefix+'_drugs',DRUG_CENTERS,BRANCH_COUNTS),
                      (prefix+'_financial_drugs',DRUG_CENTERS,FINANCIAL_BRANCH_COUNTS),
                      (prefix+'_quarter_drugs',DRUG_CENTERS,QUARTER_BRANCH_COUNTS)))
    test_payloads,test_models = [],[]
    for kind,labels,counts in specs:
        require(len(counts)==5 and len(set(counts))==5,'Нужны пять разных количеств центров.')
        require(all(isinstance(c,int) and not isinstance(c,bool) and 1<=c<=len(labels) for c in counts),
                'Число центров вне согласованного диапазона.')
        order = sorted(range(len(labels)),key=lambda i:(-len(labels[i].strip()),i))
        for count in counts:
            chosen = [labels[i] for i in order[:count]]
            if kind=='patients':
                payload = build_slide2(df,prov,month=month,labels=chosen);rows = patient_model(df,prov,chosen,month=month)
            elif kind=='procedures':
                payload = build_slide3(df,prov,labels=chosen);rows = procedure_model(df,prov,chosen)
            elif kind=='load':
                payload = build_slide4(df,prov,month=month,labels=chosen);rows = load_model(df,prov,chosen,month=month)
            elif kind=='drugs':payload,rows = build_slide5(df,prov,chosen)
            elif kind=='financial_drugs':payload,rows = build_slide6(fdf,fprov,chosen)
            elif kind=='quarter_drugs':payload,rows = build_slide7(df,prov,chosen)
            else:
                prefix,variant=kind.split('_',1)
                articles=dict(EXTRA_DRUG_PAIRS)[prefix]
                if variant=='drugs':payload,rows=build_slide5(df,prov,chosen,articles=articles)
                elif variant=='financial_drugs':payload,rows=build_slide6(fdf,fprov,chosen,articles=articles)
                else:payload,rows=build_slide7(df,prov,chosen,articles=articles)
            test_payloads.append(payload);test_models.append((kind,rows))
    return (combine_created_slides(main_payloads),combine_created_slides(test_payloads)),(main_models,test_models)


def verify_deck(payload,expectations):
    deck = Presentation(BytesIO(payload))
    require(len(deck.slides)==len(expectations),'Неверное число слайдов.')
    for slide,(kind,rows) in zip(deck.slides,expectations):
        if kind=='load':check_load_slide(slide,rows,deck.slide_width,deck.slide_height)
        elif kind.endswith('quarter_drugs'):
            articles=SLIDE5_CODES if kind=='quarter_drugs' else dict(EXTRA_DRUG_PAIRS)[kind.split('_',1)[0]]
            check_quarter_slide(slide,rows,deck.slide_width,deck.slide_height,articles=articles)
        elif kind.endswith('drugs'):
            articles=SLIDE5_CODES if kind in ('drugs','financial_drugs') else dict(EXTRA_DRUG_PAIRS)[kind.split('_',1)[0]]
            check_drug_slide(slide,rows,deck.slide_width,deck.slide_height,financial='financial' in kind,articles=articles)
        else:check_slide(slide,rows,deck.slide_width,deck.slide_height,kind=kind)
    charts = sum(2 if kind.endswith('quarter_drugs') else (4 if kind.endswith('drugs') else 1)
                 for kind,_ in expectations)
    with ZipFile(BytesIO(payload)) as package:
        require(package.testzip() is None,'Повреждён ZIP презентации.')
        require(sum(name.endswith('.xlsx') for name in package.namelist())==charts,'Независимые XLSX потеряны.')
        require(sum(name.startswith('ppt/charts/chart') and name.endswith('.xml') for name in package.namelist())==charts,
                'Неверное число независимых диаграмм.')


def save_presentations(payloads):
    if OUTPUT_DIR.resolve().is_relative_to((ROOT/'Исходники').resolve()):
        raise ValueError('Сохранение в каталог исходников запрещено.')
    require(len(payloads)==2,'Нужны два готовых пакета презентаций.')
    paths = [OUTPUT_DIR/'NL_пациенты_2026_слайд2.pptx',OUTPUT_DIR/'NL_test_branch_counts_fixed.pptx']
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    originals = [path.read_bytes() if path.exists() else None for path in paths]
    staged,replaced = [],[]
    try:
        for path,payload in zip(paths,payloads):
            from tempfile import NamedTemporaryFile
            with NamedTemporaryFile(prefix=path.stem+'-',suffix='.tmp',dir=OUTPUT_DIR,delete=False) as target:
                staged.append(Path(target.name));target.write(payload)
        for path,temporary,original in zip(paths,staged,originals):
            current = path.read_bytes() if path.exists() else None
            require(current==original,'Результат изменён другим процессом; повторите запуск.')
            os.replace(temporary,path);replaced.append((path,original))
    except Exception as error:
        for path,original in replaced:
            if original is None:path.unlink(missing_ok=True)
            else:path.write_bytes(original)
        if isinstance(error,PermissionError):
            raise PermissionError('Не удалось сохранить PPTX. Закройте целевые файлы в PowerPoint и повторите запуск.') from error
        raise
    finally:
        for temporary in staged:temporary.unlink(missing_ok=True)
    return paths


def generate_presentations(df,prov,fdf,fprov):
    payloads,expectations = build_payloads(df,prov,fdf,fprov)
    for payload,rows in zip(payloads,expectations):verify_deck(payload,rows)
    paths = save_presentations(payloads)
    return paths,[{'Файл':'основной' if i==0 else 'тестовый','Слайдов':len(rows),'Проверка':'OK'}
                  for i,rows in enumerate(expectations)]


def verify_slide5_failures(frame, provenance):
    # Малый реальный срез без пропусков; финансовые значения не выводятся.
    label = 'Актобе'
    subset = frame.loc[frame['Наименование филиала'].eq(label)
                       & frame['Код статей'].isin(SLIDE5_CODES)].copy()
    origins = provenance.loc[subset.index].copy()
    first = subset.index[subset['Месяц'].eq('январь')][0]
    for bad in (pd.NA, '#DIV/0!', -1, float('inf'), True):
        changed = subset.copy()
        changed.at[first, 'Факт по 1С'] = bad
        must_fail(ValueError, lambda: slide5_model(changed, origins, [label]))
    must_fail(ValueError, lambda: slide5_model(subset, origins, []))
    must_fail(ValueError, lambda: slide5_model(subset, origins, ['неизвестный центр']))
    must_fail(ValueError, lambda: slide5_model(subset, origins, [label,label]))
    must_fail(ValueError, lambda: slide5_model(subset, origins.iloc[::-1], [label]))
    must_fail(ValueError, lambda: slide5_model(subset, origins, [label], months=['январь','январь']))
    must_fail(ValueError, lambda: slide5_model(subset.drop(first), origins.drop(first), [label]))
    changed = subset.copy()
    changed.loc[changed['Месяц'].isin(SLIDE5_MONTHS), 'Утверждено'] = 0
    rows = slide5_model(changed, origins, [label])
    require(all(r[0]['plan']==0 for r in rows), 'Нулевой план должен сохраняться.')
    payload, expected = build_slide5(changed, origins, [label], tag='zero-check')
    verify_slide5(payload, expected)
    fractional = subset.copy()
    fractional.at[first, 'Факт по 1С'] = Decimal('.125')
    expected = slide5_model(fractional, origins, [label])
    require(expected[0][0]['actual'] % 1 == Decimal('.125'), 'Дробный расход потерян.')
    for metric in ('Факт по норме','Факт по 1С'):
        for invalid in (pd.NA,'#DIV/0!',-1,float('inf'),True):
            changed=subset.copy();changed.at[first,metric]=invalid
            must_fail(ValueError,lambda:slide5_model(changed,origins,[label]))
    for delta in (-1,0,1):
        changed=subset.copy()
        changed.loc[changed['Месяц'].isin(SLIDE5_MONTHS),'Утверждено']=1000
        changed.loc[changed['Месяц'].isin(SLIDE5_MONTHS),'Факт по норме']=10+delta
        changed.loc[changed['Месяц'].isin(SLIDE5_MONTHS),'Факт по 1С']=10
        payload,models=build_slide5(changed,origins,[label])
        require(all(r[0]['difference']==Decimal(delta)*len(SLIDE5_MONTHS) for r in models),
                'Отклонение должно рассчитываться по факту по норме.')
        verify_slide5(payload,models)
    duplicate = subset.loc[[first]].copy()
    duplicate.index = [max(subset.index)+1]
    duplicate_origin = origins.loc[[first]].copy()
    duplicate_origin.index = duplicate.index
    must_fail(ValueError, lambda: slide5_model(pd.concat([subset,duplicate]),
                                             pd.concat([origins,duplicate_origin]),[label]))
    print('PASS slide5 failures: missing/error/negative/nonfinite, months, centers, duplicates; zero and fractional values.')



def verify_source_values(df,prov):
    workbook = load_workbook(excel_files['quantity'],read_only=True,data_only=True)
    cache = {}
    try:
        for branch in BRANCH_SHEETS:
            values = list(workbook[branch].iter_rows(min_row=6,max_row=48,max_col=43,values_only=True))
            columns = {header_text(v):i for i,v in enumerate(values[0]) if header_text(v) in MONTHS[:9]}
            cache[branch] = (values,columns)
        def summed(row,number,months,key,article=None):
            values,columns = cache[row['branch']];total = Decimal(0)
            for month in months:
                col = columns[month]+{'plan':0,'norm':1,'actual':2}[key]
                require(header_text(values[1][col])=={'plan':'утвержд.','norm':'факт по норме','actual':'факт по 1с'}[key],
                        'Неожиданный заголовок Excel.')
                value = values[number-6][col]
                if value is None and row['branch']=='Атырау Нурсая' and number in (47,48) and month=='август' and key=='actual':
                    value = 0
                require(isinstance(value,(int,float)) and not isinstance(value,bool),'Нет исходного числа Excel.')
                total += Decimal(str(value))
            if article is not None:require(values[number-6][2]==article,'Статья Excel изменилась.')
            return total
        patients = patient_model(df,prov,PATIENT_CENTERS,month=REPORT_MONTH)
        procedures = procedure_model(df,prov,PROCEDURE_CENTERS)
        for rows,number,months in [(patients,14,[REPORT_MONTH]),(procedures,15,MONTHS[:9])]:
            for row in rows:
                for key in ('plan','actual'):
                    require(row[key]==summed(row,number,months,key),'Сумма не совпала с независимым чтением Excel.')
        for row in load_model(df,prov,LOAD_CENTERS,month=REPORT_MONTH):
            for key in ('plan','actual'):
                numerator,denominator = summed(row,14,[REPORT_MONTH],key),summed(row,13,[REPORT_MONTH],key)
                require(row[key]==numerator/denominator,'Неверная нагрузка по исходным ячейкам.')
        for article,rows,number in zip(SLIDE5_ARTICLES,slide5_model(df,prov,DRUG_CENTERS),(47,48)):
            for row in rows:
                for key in ('plan','norm','actual'):
                    require(row[key]==summed(row,number,SLIDE5_MONTHS,key,article),'Неверный девятимесячный расход.')
                require(row['difference']==summed(row,number,SLIDE5_MONTHS,'norm',article)-summed(row,number,SLIDE5_MONTHS,'actual',article),
                        'Отклонение не совпало с фактом по норме минус фактом 1С.')
    finally:workbook.close()
    print('PASS: независимые месячные значения Excel для пациентов, процедур, нагрузки и препаратов.')


def verify_model_failures(df,prov):
    for factory,labels,options in [(patient_model,PATIENT_CENTERS,{'month':REPORT_MONTH}),
                                  (procedure_model,PROCEDURE_CENTERS,{}),
                                  (load_model,LOAD_CENTERS,{'month':REPORT_MONTH})]:
        rows = factory(df,prov,labels,**options)
        index = rows[0].get('source_index',rows[0].get('apparatus_index'))
        for invalid in (pd.NA,'#DIV/0!',-1,float('inf'),True):
            changed = df.copy();changed.at[index,'Факт по 1С'] = invalid
            must_fail(ValueError,lambda:factory(changed,prov,labels,**options))
        must_fail(ValueError,lambda:factory(df,prov.iloc[::-1],labels,**options))
        must_fail(ValueError,lambda:factory(df,prov,[],**options))
        must_fail(ValueError,lambda:factory(df,prov,[labels[0],labels[0]],**options))
    for factory,labels,index_field in [(patient_model,PATIENT_CENTERS,'source_index'),(load_model,LOAD_CENTERS,'apparatus_index')]:
        index = factory(df,prov,labels,month=REPORT_MONTH)[0][index_field]
        changed = df.copy();changed.at[index,'Факт по 1С'] = Decimal('1.5')
        must_fail(ValueError,lambda:factory(changed,prov,labels,month=REPORT_MONTH))
    selected = procedure_model(df,prov,PROCEDURE_CENTERS)[0]['source_indices'];index = selected[0]
    changed = df.drop(index);must_fail(ValueError,lambda:procedure_model(changed,prov.drop(index),PROCEDURE_CENTERS))
    repeated = pd.concat([df,df.loc[[index]]],ignore_index=True)
    repeated_prov = pd.concat([prov,prov.loc[[index]]],ignore_index=True)
    must_fail(ValueError,lambda:procedure_model(repeated,repeated_prov,PROCEDURE_CENTERS))
    changed = df.copy();changed.loc[list(selected),'Утверждено'] = 0
    must_fail(ValueError,lambda:procedure_model(changed,prov,PROCEDURE_CENTERS))
    changed = df.copy();changed.loc[changed['Месяц'].isin(MONTHS[9:]),['Утверждено','Факт по 1С']] = '#DIV/0!'
    require(procedure_model(changed,prov,PROCEDURE_CENTERS)==procedure_model(df,prov,PROCEDURE_CENTERS),
            'Включены октябрь–декабрь.')
    apparatus = load_model(df,prov,LOAD_CENTERS,month=REPORT_MONTH)[0]['apparatus_index']
    changed = df.copy();changed.at[apparatus,'Факт по 1С'] = 0
    must_fail(ValueError,lambda:load_model(changed,prov,LOAD_CENTERS,month=REPORT_MONTH))
    verify_slide5_failures(df,prov)
    print('PASS: неверные числа, месяцы, индексы, дубли, знаменатели и исключение месяцев вне периода.')


def verify_cold_start(df,prov,fdf,fprov):
    from unittest.mock import patch
    original_presentation,real_load = Presentation,load_workbook
    changed = df.copy()
    index = patient_model(df,prov,PATIENT_CENTERS,month=REPORT_MONTH)[0]['source_index']
    changed.at[index,'Факт по 1С'] = int(changed.at[index,'Факт по 1С'])+1
    drug_index=slide5_model(df,prov,DRUG_CENTERS)[0][0]['source_indices'][0]
    changed.at[drug_index,'Факт по норме']=Decimal(str(changed.at[drug_index,'Факт по норме']))+1
    changed_fin=fdf.copy()
    fin_index=slide6_model(fdf,fprov,DRUG_CENTERS)[0][0]['source_indices'][0]
    changed_fin.at[fin_index,'Факт по норме']=Decimal(str(changed_fin.at[fin_index,'Факт по норме']))+Decimal(1000)
    def forbidden(*args,**kwargs):
        raise AssertionError('Генерация попыталась прочитать входной PPTX/бюджет или повторить импорт.')
    def standalone_presentation(source=None):
        if source is not None and not isinstance(source,BytesIO):forbidden()
        return original_presentation(source)
    def reference_only(source,*args,**kwargs):
        if isinstance(source,BytesIO):return real_load(source,*args,**kwargs)
        if Path(source).resolve()!=CENTERS.resolve():forbidden()
        return real_load(source,*args,**kwargs)
    with TemporaryDirectory(prefix='papagun-cold-start-') as temporary:
        empty_output = Path(temporary)/'new-output'
        with patch.dict(globals(),{'OUTPUT_DIR':empty_output,'Presentation':standalone_presentation,
                                  'load_workbook':reference_only,'workbooks':None,'quantity_df':None,
                                  'financial_df':None,'quantity_sheets':None,'financial_sheets':None,
                                  'build_branches_dataframe':forbidden}),patch.object(pd,'read_excel',side_effect=forbidden):
            paths,summary = generate_presentations(changed,prov,changed_fin,fprov)
            require([r['Слайдов'] for r in summary]==[18,90],'Холодный запуск не создал 18/90 слайдов.')
            first = [path.read_bytes() for path in paths]
            paths,summary = generate_presentations(changed,prov,changed_fin,fprov)
            require(len(list(empty_output.iterdir()))==2,'Повторный запуск создал лишние файлы.')
            repeated = [path.read_bytes() for path in paths]
        real_replace = os.replace
        attempts = [0]
        def fail_second_replace(source,target):
            attempts[0] += 1
            if attempts[0]==2:raise PermissionError('Проверка отката второго файла.')
            return real_replace(source,target)
        with patch.dict(globals(),{'OUTPUT_DIR':empty_output}),patch.object(os,'replace',side_effect=fail_second_replace):
            must_fail(PermissionError,lambda:save_presentations(first))
        require([path.read_bytes() for path in paths]==repeated,'Ошибка записи повредила прежние результаты.')
        require(len(list(empty_output.iterdir()))==2,'После ошибки остались временные файлы.')
        with patch.dict(globals(),{'OUTPUT_DIR':ROOT/'Исходники'}):
            must_fail(ValueError,lambda:save_presentations(first))
        for left,right in zip(first,repeated):
            a,b = original_presentation(BytesIO(left)),original_presentation(BytesIO(right))
            require([str(s._element.xml) for s in a.slides]==[str(s._element.xml) for s in b.slides],
                    'Повторный запуск изменил структуру/создал дубликаты.')
        result = original_presentation(BytesIO(first[0]))
        chart = next(s.chart for s in result.slides[0].shapes if s.has_chart)
        expected = patient_model(changed,prov,PATIENT_CENTERS,month=REPORT_MONTH)
        require(list(chart.series[1].values)==[float(r['actual']) for r in expected],
                'Генерация проигнорировала переданный DataFrame.')
        financial_models=slide6_model(changed_fin,fprov,DRUG_CENTERS)
        check_drug_slide(result.slides[4],financial_models,result.slide_width,result.slide_height,financial=True)
        require(financial_models[0][0]['norm']-slide6_model(fdf,fprov,DRUG_CENTERS)[0][0]['norm']==1,
                'Изменение переданного financial_df потеряно.')
        quarter_models=slide7_model(changed,prov,DRUG_CENTERS)
        check_quarter_slide(result.slides[5],quarter_models,result.slide_width,result.slide_height)
        require(quarter_models[0][0]['norm']-slide7_model(df,prov,DRUG_CENTERS)[0][0]['norm']==1,
                'Кварталы проигнорировали переданный quantity_df.')
    print('PASS: пустая папка, входные PPTX и повторное чтение бюджетов запрещены; 18/90 слайдов, повторный запуск, откат ошибки записи и защита исходников.')


def verify_financial_drugs(frame,provenance):
    original=frame.copy(deep=True)
    models=slide6_model(frame,provenance,DRUG_CENTERS)
    workbook=load_workbook(excel_files['financial'],read_only=True,data_only=True)
    try:
        cache={branch:list(workbook[branch].iter_rows(min_row=6,max_row=48,max_col=43,values_only=True))
               for branch in {r['branch'] for rows in models for r in rows}}
        for article,rows,number in zip(SLIDE5_ARTICLES,models,(47,48)):
            for row in rows:
                cells=cache[row['branch']]
                require(cells[number-6][2]==article,'Изменилась финансовая статья Excel.')
                columns={header_text(v):j for j,v in enumerate(cells[0]) if header_text(v) in SLIDE6_MONTHS}
                expected={}
                for key,offset,header in [('norm',1,'факт по норме'),('actual',2,'факт по 1с')]:
                    total=Decimal(0)
                    for month in SLIDE6_MONTHS:
                        col=columns[month]+offset
                        require(header_text(cells[1][col])==header,'Неверная финансовая колонка.')
                        value=cells[number-6][col]
                        if value is None and key=='actual' and month=='сентябрь':value=0
                        require(isinstance(value,(int,float)) and not isinstance(value,bool),'Отсутствует финансовое число Excel.')
                        total+=Decimal(str(value))
                    expected[key]=total/Decimal(1000)
                require(row['plan']==row['norm']==expected['norm'] and row['actual']==expected['actual'],
                        'Финансовая сумма или перевод в тыс. тенге не совпали с Excel.')
                require(row['difference']==expected['norm']-expected['actual'],'Неверное финансовое отклонение.')
    finally:workbook.close()
    payload,models=build_slide6(frame,provenance)
    deck=Presentation(BytesIO(payload))
    check_drug_slide(deck.slides[0],models,deck.slide_width,deck.slide_height,financial=True)
    subset=frame.loc[frame['Наименование филиала'].eq('Актобе') & frame['Код статей'].isin(SLIDE5_CODES)].copy()
    origins=provenance.loc[subset.index].copy()
    index=subset.index[subset['Месяц'].eq('январь')][0]
    for metric in ('Факт по норме','Факт по 1С'):
        for bad in (pd.NA,'#DIV/0!',-1,float('inf'),True):
            changed=subset.copy();changed.at[index,metric]=bad
            must_fail(ValueError,lambda:slide6_model(changed,origins,['Актобе']))
    september=subset.index[subset['Месяц'].eq('сентябрь')][0]
    changed=subset.copy();changed.at[september,'Факт по норме']=pd.NA
    must_fail(ValueError,lambda:slide6_model(changed,origins,['Актобе']))
    changed=subset.copy();changed['Утверждено']=pd.NA
    require(slide6_model(changed,origins,['Актобе'])==slide6_model(subset,origins,['Актобе']),
            'Финансовый слайд использует Утверждено.')
    must_fail(ValueError,lambda:slide6_model(subset.drop(index),origins.drop(index),['Актобе']))
    must_fail(ValueError,lambda:slide6_model(subset,origins,[]))
    must_fail(ValueError,lambda:slide6_model(subset,origins,['Актобе','Актобе']))
    must_fail(ValueError,lambda:slide6_model(subset,origins.assign(Книга='quantity'),['Актобе']))
    must_fail(ValueError,lambda:slide6_model(subset,origins,['Актобе'],months=['январь','январь']))
    repeated=pd.concat([subset,subset.loc[[index]]],ignore_index=True)
    repeated_origin=pd.concat([origins,origins.loc[[index]]],ignore_index=True)
    must_fail(ValueError,lambda:slide6_model(repeated,repeated_origin,['Актобе']))
    for delta in (Decimal('-0.125'),Decimal(0),Decimal('0.125')):
        changed=subset.copy()
        changed.loc[changed['Месяц'].isin(SLIDE6_MONTHS),'Факт по норме']=Decimal(10)+delta
        changed.loc[changed['Месяц'].isin(SLIDE6_MONTHS),'Факт по 1С']=Decimal(10)
        payload,rows=build_slide6(changed,origins,['Актобе'])
        require(all(r[0]['difference']==delta*9/1000 for r in rows),'Финансовые дроби или знак потеряны.')
        deck=Presentation(BytesIO(payload))
        check_drug_slide(deck.slides[0],rows,deck.slide_width,deck.slide_height,financial=True)
    require(drug_execution_text(Decimal(201),Decimal(200))=='101%','Процент не округляется до целого.')
    require(drug_execution_text(Decimal(0),Decimal(0))=='н/д','Нулевой знаменатель не должен давать процент.')
    require(drug_execution_text(Decimal(1),Decimal(0))=='н/д','Нулевой факт 1С не должен давать процент.')
    require(drug_execution_text(Decimal(0),Decimal(1))=='0%','Нулевая норма должна давать 0% при ненулевом факте.')
    # Отношение сумм равно 250%, средний процент двух центров был бы 200%.
    synthetic=[[dict(label='A',article=name,article_code=code,norm=Decimal(1),plan=Decimal(1),actual=Decimal(1),difference=Decimal(0)),
                dict(label='B',article=name,article_code=code,norm=Decimal(9),plan=Decimal(9),actual=Decimal(3),difference=Decimal(6))]
               for code,name in zip(SLIDE5_CODES,SLIDE5_ARTICLES)]
    payload=render_drug_models(synthetic,heading='РАСХОДЫ ЛС: январь–сентябрь 2026 г. (тыс. тенге)',
                              series_names=('Факт по норме','Факт по 1С','отк:'),financial=True)
    test_deck=Presentation(BytesIO(payload))
    panels=[sh for sh in test_deck.slides[0].shapes if sh.name.startswith('drug_execution:')]
    require(len(panels)==2 and all(sh.text=='250%' for sh in panels),'В панели усреднены проценты филиалов.')
    require(frame.equals(original),'financial_df изменён при обработке пропусков.')
    print('PASS financial drugs: Excel, тыс. тенге, 9 месяцев, разрешённые нули, знаки/цвета, ошибки и неизменность DataFrame.')


def check_quarter_slide(slide,models,width,height,*,articles=SLIDE5_CODES):
    require(any(sh.has_text_frame and sh.text=='РАСХОДЫ ЛС: по кварталам 2026 г. (количество)' for sh in slide.shapes),
            'Неверный квартальный заголовок.')
    charts=sorted((sh for sh in slide.shapes if sh.has_chart),key=lambda sh:sh.top)
    require(len(charts)==2,'Нужны две квартальные диаграммы.')
    require(charts[0].height>=Inches(3.09) and charts[1].height>=Inches(3.11)
            and charts[0].top+charts[0].height<charts[1].top,
            'Квартальные диаграммы слишком низкие или пересекаются.')
    for article_code,rows,shape in zip(articles,models,charts):
        article=rows[0]['article']
        require(all(r['article_code']==article_code and r['article']==article for r in rows),
                'Неверный код или наименование квартальной модели.')
        require([r['label'] for r in rows]==['1 квартал','2 квартал','3 квартал','9 месяцев'],'Неверные квартальные периоды.')
        require(rows[3]['norm']==sum((r['norm'] for r in rows[:3]),Decimal(0))
                and rows[3]['actual']==sum((r['actual'] for r in rows[:3]),Decimal(0)),
                'Девять месяцев не равны сумме трёх кварталов.')
        titles=[sh for sh in slide.shapes if sh.name=='quarter_article:'+article]
        require(len(titles)==1 and titles[0].text==article,'Неверная квартальная статья.')
        title=titles[0]
        require(title.text_frame.paragraphs[0].alignment==PP_ALIGN.LEFT,'Квартальная статья должна быть выровнена влево.')
        require(shape.left<=title.left and title.left+title.width<=shape.left+shape.width
                and shape.top<=title.top and title.top+title.height<=shape.top+shape.height,'Статья вне рамки.')
        chart=shape.chart
        layout=chart._chartSpace.xpath('.//c:plotArea/c:layout/c:manualLayout')[0]
        require(float(layout.xpath('./c:h')[0].get('val'))>=.66,
                'Внутренняя область квартальной диаграммы слишком низкая.')
        require([se.name for se in chart.series]==['Факт по норме','Факт по 1С','отк:'],'Неверные квартальные серии.')
        require([c.label for c in chart.plots[0].categories]==[r['label'] for r in rows],'Неверные квартальные категории.')
        for se,key in zip(chart.series,('norm','actual','difference')):
            require(list(se.values)==[float(r[key]) for r in rows],'Неверные квартальные значения.')
            require(se.data_labels.number_format=='#,##0','Квартальные метки должны быть целыми.')
            if key=='difference':
                for point,row in zip(se.points,rows):
                    value=row[key]
                    require(str(point.format.fill.fore_color.rgb).upper()==(STYLE['red'] if value<0 else STYLE['gray']),
                            'Неверная заливка квартального отклонения.')
                    color=STYLE['red'] if value<0 else (STYLE['green'] if value>0 else '000000')
                    require(str(point.data_label.font.color.rgb).upper()==color,'Неверный цвет квартальной метки.')
        workbook=load_workbook(BytesIO(chart.part.chart_workbook.xlsx_part.blob),read_only=True,data_only=True)
        try:
            entries=list(workbook.active.values)
            require(len(entries)==5,'Неверная квартальная XLSX.')
            for entry,row in zip(entries[1:],rows):
                require(entry[0]==row['label'],'Неверная категория XLSX.')
                for value,key in zip(entry[1:],('norm','actual','difference')):
                    require(abs(float(value)-float(row[key]))<1e-8,'Неверное квартальное значение XLSX.')
        finally:workbook.close()
        for row in rows:
            panels=[sh for sh in slide.shapes if sh.name=='quarter_execution:'+article+':'+row['label']]
            expected='н/д' if row['norm']==0 else str((row['actual']/row['norm']*100).quantize(Decimal(1),rounding=ROUND_HALF_UP))+'%'
            require(len(panels)==1 and panels[0].text==expected,'Неверный квартальный процент.')
            panel=panels[0]
            expected_color=STYLE['red'] if row['norm']>0 and row['actual']>row['norm'] else '000000'
            require(all(str(run.font.color.rgb).upper()==expected_color for para in panel.text_frame.paragraphs for run in para.runs),
                    'Неверный цвет квартального процента относительно 100%.')
            require(shape.left<=panel.left and panel.left+panel.width<=shape.left+shape.width
                    and shape.top<=panel.top and panel.top+panel.height<=shape.top+shape.height,'Квартальная панель вне рамки.')
    require(not any(sh.name=='quarter_centers' or (sh.has_text_frame and sh.text.startswith('Сумма по показанным центрам')) for sh in slide.shapes),
            'Нижняя подпись квартального слайда должна быть удалена.')
    for sh in slide.shapes:
        require(0<=sh.left and 0<=sh.top and sh.left+sh.width<=width and sh.top+sh.height<=height,'Квартальная фигура вне слайда.')


def verify_quarter_drugs(frame,provenance):
    original=frame.copy(deep=True);models=slide7_model(frame,provenance,DRUG_CENTERS)
    mapping=slide5_center_map();branches=[mapping[_center_key(label)] for label in DRUG_CENTERS]
    workbook=load_workbook(excel_files['quantity'],read_only=True,data_only=True)
    try:
        cache={branch:list(workbook[branch].iter_rows(min_row=6,max_row=48,max_col=43,values_only=True)) for branch in branches}
        for article,rows,number in zip(SLIDE5_ARTICLES,models,(47,48)):
            for row in rows:
                expected={}
                for key,offset,header in [('norm',1,'факт по норме'),('actual',2,'факт по 1с')]:
                    total=Decimal(0)
                    for branch in branches:
                        cells=cache[branch]
                        require(cells[number-6][2]==article,'Изменилась квартальная статья Excel.')
                        columns={header_text(v):i for i,v in enumerate(cells[0]) if header_text(v) in SLIDE5_MONTHS}
                        for month in row['months']:
                            col=columns[month]+offset
                            require(header_text(cells[1][col])==header,'Неверная квартальная колонка Excel.')
                            value=cells[number-6][col]
                            if value is None and branch=='Атырау Нурсая' and month=='август' and key=='actual':value=0
                            require(isinstance(value,(int,float)) and not isinstance(value,bool),'Нет квартального исходного числа.')
                            total+=Decimal(str(value))
                    expected[key]=total
                require(row['norm']==expected['norm'] and row['actual']==expected['actual']
                        and row['difference']==expected['norm']-expected['actual'],'Кварталы не совпали с независимым Excel.')
    finally:workbook.close()
    payload,models=build_slide7(frame,provenance);deck=Presentation(BytesIO(payload))
    check_quarter_slide(deck.slides[0],models,deck.slide_width,deck.slide_height)
    subset=frame.loc[frame['Наименование филиала'].eq('Актобе') & frame['Код статей'].isin(SLIDE5_CODES)].copy()
    origins=provenance.loc[subset.index];first=subset.index[subset['Месяц'].eq('январь')][0]
    base=slide7_model(subset,origins,['Актобе'])
    for metric in ('Факт по норме','Факт по 1С'):
        for bad in (pd.NA,'#DIV/0!',-1,float('inf'),True):
            changed=subset.copy();changed.at[first,metric]=bad
            must_fail(ValueError,lambda:slide7_model(changed,origins,['Актобе']))
    must_fail(ValueError,lambda:slide7_model(subset.drop(first),origins.drop(first),['Актобе']))
    must_fail(ValueError,lambda:slide7_model(subset,origins,[]))
    must_fail(ValueError,lambda:slide7_model(subset,origins,['Актобе','Актобе']))
    duplicate=pd.concat([subset,subset.loc[[first]]],ignore_index=True)
    duplicate_origin=pd.concat([origins,origins.loc[[first]]],ignore_index=True)
    must_fail(ValueError,lambda:slide7_model(duplicate,duplicate_origin,['Актобе']))
    changed=subset.copy();changed.at[first,'Факт по 1С']=Decimal(str(changed.at[first,'Факт по 1С']))+Decimal('.125')
    result=slide7_model(changed,origins,['Актобе'])
    require(result[0][0]['actual']-base[0][0]['actual']==Decimal('.125')
            and result[0][3]['actual']-base[0][3]['actual']==Decimal('.125')
            and result[0][1:3]==base[0][1:3],'Квартальные границы или дроби потеряны.')
    changed=subset.copy();changed['Утверждено']=pd.NA
    changed.loc[~changed['Месяц'].isin(SLIDE5_MONTHS),['Факт по норме','Факт по 1С']]='#DIV/0!'
    require(slide7_model(changed,origins,['Актобе'])==base,'Кварталы используют Утверждено или четвёртый квартал.')
    for delta in (-1,0,1):
        changed=subset.copy()
        changed.loc[changed['Месяц'].isin(SLIDE5_MONTHS),'Факт по норме']=10+delta
        changed.loc[changed['Месяц'].isin(SLIDE5_MONTHS),'Факт по 1С']=10
        payload,rows=build_slide7(changed,origins,['Актобе']);deck=Presentation(BytesIO(payload))
        require(all(r['difference']==Decimal(delta)*(3 if j<3 else 9) for part in rows for j,r in enumerate(part)),
                'Неверная квартальная формула отклонения.')
        check_quarter_slide(deck.slides[0],rows,deck.slide_width,deck.slide_height)
    require(quarter_execution_info(Decimal(100),Decimal(100))==('100%','000000'),'100% должны быть чёрными.')
    require(quarter_execution_info(Decimal(100),Decimal('100.1'))==('100%',STYLE['red']),
            'Цвет зависит от исходного процента, а не округлённой подписи.')
    require(quarter_execution_info(Decimal(100),Decimal('99.9'))==('100%','000000'),'Процент ниже 100 не должен быть красным.')
    require(quarter_execution_info(Decimal(0),Decimal(1))==('н/д','000000'),'Нулевой знаменатель квартального процента.')
    require(frame.equals(original),'quantity_df изменён квартальной моделью.')
    print('PASS quarterly drugs: кварталы и 9 месяцев, Excel, нормы/факты/разности, проценты, XLSX, границы, ошибки и цвета.')


def verify_article_pair(qdf,qprov,fdf,fprov,articles,source_rows):
    """Check the new article pair against source cells and the three rendered slides."""
    for book,frame,provenance,model,builder in (
        ('quantity',qdf,qprov,slide5_model,build_slide5),
        ('financial',fdf,fprov,slide6_model,build_slide6),
    ):
        original=frame.copy(deep=True)
        models=model(frame,provenance,DRUG_CENTERS,articles=articles)
        workbook=load_workbook(excel_files[book],read_only=True,data_only=True)
        try:
            for article_code,rows,number in zip(articles,models,source_rows):
                for row in rows:
                    cells=list(workbook[row['branch']].iter_rows(min_row=6,max_row=number,max_col=43,values_only=True))
                    require(cells[number-6][1]==article_code and cells[number-6][2]==row['article'],
                            'Код или наименование Excel не совпали с моделью слайда.')
                    columns={header_text(v):j for j,v in enumerate(cells[0]) if header_text(v) in SLIDE5_MONTHS}
                    totals={}
                    for key,offset,header in (
                        ('plan',0 if book=='quantity' else 1,'утвержд.' if book=='quantity' else 'факт по норме'),
                        ('norm',1,'факт по норме'),('actual',2,'факт по 1с')):
                        total=Decimal(0)
                        for month in SLIDE5_MONTHS:
                            col=columns[month]+offset
                            require(header_text(cells[1][col])==header,'Неверная колонка новой статьи Excel.')
                            value=cells[number-6][col]
                            if (value is None and key=='actual' and
                                ((book=='quantity' and row['branch']=='Атырау Нурсая' and month=='август')
                                 or (book=='financial' and month=='сентябрь'))):
                                value=0
                            require(isinstance(value,(int,float)) and not isinstance(value,bool) and value>=0,
                                    'Для новой статьи нет допустимого числа Excel.')
                            total+=Decimal(str(value))
                        totals[key]=total/(FINANCIAL_DISPLAY_DIVISOR if book=='financial' else Decimal(1))
                    require(all(row[key]==totals[key] for key in totals),'Новая статья не совпала с Excel.')
                    require(row['difference']==totals['norm']-totals['actual'],'Неверное отклонение новой статьи.')
                    require(all(provenance.loc[i,'Строка Excel']==number for i in row['source_indices']),
                            'Новая статья взята не из заданной строки Excel.')
        finally:workbook.close()
        payload,expected=builder(frame,provenance,articles=articles)
        deck=Presentation(BytesIO(payload))
        check_drug_slide(deck.slides[0],expected,deck.slide_width,deck.slide_height,
                         financial=book=='financial',articles=articles)
        subset=frame.loc[frame['Наименование филиала'].eq(models[0][0]['branch'])
                         & frame['Код статей'].isin(articles)].copy()
        origins=provenance.loc[subset.index]
        first=subset.index[subset['Месяц'].eq('январь')][0]
        subset.at[first,'Факт по 1С']=pd.NA
        must_fail(ValueError,lambda:model(subset,origins,[models[0][0]['label']],articles=articles))
        require(frame.equals(original),'Новые статьи изменили DataFrame.')
    quarterly=slide7_model(qdf,qprov,DRUG_CENTERS,articles=articles)
    for article_index,rows in enumerate(quarterly):
        monthly=slide5_model(qdf,qprov,DRUG_CENTERS,articles=articles)[article_index]
        require(rows[-1]['norm']==sum((r['norm'] for r in monthly),Decimal(0))
                and rows[-1]['actual']==sum((r['actual'] for r in monthly),Decimal(0)),
                'Квартальный итог новой статьи не совпал с месячными данными.')
    payload,expected=build_slide7(qdf,qprov,articles=articles)
    deck=Presentation(BytesIO(payload))
    check_quarter_slide(deck.slides[0],expected,deck.slide_width,deck.slide_height,articles=articles)
    print('PASS article pair: Excel rows',source_rows,'quantity, financial, quarters, PPTX/XLSX and unchanged DataFrames.')


def verify_code_selection(qdf,qprov,fdf,fprov):
    """Names label slides; codes, with source rows for load, select the values."""
    baseline=load_model(qdf,qprov,LOAD_CENTERS,month=REPORT_MONTH)
    changed=qdf.copy()
    selected=qprov['Строка Excel'].isin((13,14))
    changed.loc[selected,'Наименование статей']='Переименовано для проверки'
    require(load_model(changed,qprov,LOAD_CENTERS,month=REPORT_MONTH)==baseline,
            'Нагрузка выбрана по наименованию вместо кода.')
    bad=changed.copy()
    bad.at[baseline[0]['apparatus_index'],'Код статей']='999'
    must_fail(ValueError,lambda:load_model(bad,qprov,LOAD_CENTERS,month=REPORT_MONTH))
    for frame,provenance,model,builder in ((qdf,qprov,slide5_model,build_slide5),
                                            (fdf,fprov,slide6_model,build_slide6)):
        labels=['Актобе']
        baseline=model(frame,provenance,labels,articles=PARICALCITOL_DRUG_CODES)
        renamed=frame.copy()
        mask=renamed['Наименование филиала'].eq('Актобе') & renamed['Код статей'].eq('211.10')
        renamed.loc[mask,'Наименование статей']='Проверочное наименование статьи'
        rows=model(renamed,provenance,labels,articles=PARICALCITOL_DRUG_CODES)
        for old,new in zip(baseline,rows):
            require(all(old[0][key]==new[0][key] for key in ('plan','norm','actual','difference')),
                    'Переименование изменило расчёт по коду.')
        require(rows[1][0]['article']=='Проверочное наименование статьи',
                'Модель не взяла подпись из Наименования статей.')
        payload,_=builder(renamed,provenance,labels,articles=PARICALCITOL_DRUG_CODES)
        deck=Presentation(BytesIO(payload))
        require(any(sh.has_text_frame and sh.text=='Проверочное наименование статьи' for sh in deck.slides[0].shapes),
                'Слайд не показывает наименование из выбранной книги.')
        bad=renamed.copy()
        bad.at[rows[1][0]['source_indices'][0],'Код статей']='999'
        must_fail(ValueError,lambda:model(bad,provenance,labels,articles=PARICALCITOL_DRUG_CODES))
    print('PASS code selection: names label slides; code changes block load and drug models.')


## Запуск


In [ ]:
report_path = test_pptx_path = None
report_summary = []
source_files = [CENTERS,*excel_files.values()]
source_hashes_before = {path:digest(path) for path in source_files}
if RUN_CHECKS:
    verify_source_values(quantity_df,quantity_provenance)
    verify_model_failures(quantity_df,quantity_provenance)
    verify_financial_drugs(financial_df,financial_provenance)
    verify_quarter_drugs(quantity_df,quantity_provenance)
    verify_article_pair(quantity_df,quantity_provenance,financial_df,financial_provenance,NEW_DRUG_CODES,(49,50))
    verify_article_pair(quantity_df,quantity_provenance,financial_df,financial_provenance,MINERAL_DRUG_CODES,(54,55))
    verify_article_pair(quantity_df,quantity_provenance,financial_df,financial_provenance,PARICALCITOL_DRUG_CODES,(53,56))
    verify_article_pair(quantity_df,quantity_provenance,financial_df,financial_provenance,CALDEROL_DRUG_CODES,(57,58))
    verify_code_selection(quantity_df,quantity_provenance,financial_df,financial_provenance)
    verify_cold_start(quantity_df,quantity_provenance,financial_df,financial_provenance)
if GENERATE_PRESENTATIONS:
    (report_path,test_pptx_path),report_summary = generate_presentations(quantity_df,quantity_provenance,financial_df,financial_provenance)
    print('PASS: самостоятельно созданы основной отчёт (18 слайдов) и тестовый (90 слайдов).')
require(all(digest(path)==value for path,value in source_hashes_before.items()),'Исходные Excel изменились.')
pd.DataFrame(report_summary)


## Необязательный просмотр через PowerPoint
Вся реализация локального рендера также находится в notebook. Она требует Windows и установленного PowerPoint, открывает PPTX только для чтения, проверяет рамки текста и число строк числовых подписей. При включении превью проверяются итоговая и тестовая презентации. Можно экспортировать все слайды или выбранные номера (`slides=[2]` для образца). Пути передаются через переменные окружения, не вставляются в код PowerShell.
По умолчанию `RENDER_PREVIEWS=False`. При включении PNG и JSON создаются во временной папке, картинки показываются в notebook, папка затем удаляется. Перед сохранением очищайте outputs. Постоянные PNG и подпапки output не создаются.


In [ ]:
def render_presentation(presentation, directory, *, slides=None):
    """Render locally via installed PowerPoint; no external project scripts."""
    source = Path(presentation).resolve(strict=True)
    target = Path(directory).resolve()
    if target.exists():
        raise FileExistsError('Выберите новый каталог превью.')
    if slides is not None and (not slides or any(not isinstance(n, int) or n < 1 for n in slides)):
        raise ValueError('Номера слайдов должны быть положительными целыми числами.')
    script = r"""
$ErrorActionPreference = 'Stop'
$source = (Resolve-Path -LiteralPath $env:PAPAGUN_RENDER_SOURCE).Path
$beforeHash = (Get-FileHash -LiteralPath $source -Algorithm SHA256).Hash
$target = [System.IO.Path]::GetFullPath($env:PAPAGUN_RENDER_DIR)
if (Test-Path -LiteralPath $target) { throw 'Preview directory already exists; select another Directory.' }
[System.IO.Directory]::CreateDirectory($target) | Out-Null
$app = $null
$deck = $null
$results = @()
try {
    $app = New-Object -ComObject PowerPoint.Application
    $deck = $app.Presentations.Open($source, -1, 0, 0)
    $indices = if ($env:PAPAGUN_RENDER_SLIDES) { @($env:PAPAGUN_RENDER_SLIDES.Split(',') | ForEach-Object { [int]$_ }) } else { @(1..$deck.Slides.Count) }
    foreach ($index in $indices) {
        if ($index -lt 1 -or $index -gt $deck.Slides.Count) { throw 'Invalid slide number.' }
        $slide = $deck.Slides.Item($index)
        $slide.Export((Join-Path $target "slide$index.png"), 'PNG', 1920, 1080)
        $overflow = @()
        $wrappedLabels = @()
        foreach ($shape in $slide.Shapes) {
            if ($shape.HasTextFrame -eq -1 -and $shape.TextFrame2.HasText -eq -1) {
                if ($shape.TextFrame2.TextRange.BoundWidth -gt ($shape.Width + 1) -or
                    $shape.TextFrame2.TextRange.BoundHeight -gt ($shape.Height + 1)) {
                    $overflow += [int]$shape.Id
                }
            }
            if ($shape.HasChart -eq -1) {
                $chart = $shape.Chart
                for ($seriesIndex = 1; $seriesIndex -le $chart.SeriesCollection().Count; $seriesIndex++) {
                    $series = $chart.SeriesCollection($seriesIndex)
                    for ($pointIndex = 1; $pointIndex -le $series.Points().Count; $pointIndex++) {
                        $point = $series.Points($pointIndex)
                        # Rotated labels are tall even when they contain a single line.
                        if ($point.HasDataLabel -and $point.DataLabel.Format.TextFrame2.TextRange.Lines().Count -gt 1) {
                            $wrappedLabels += [pscustomobject]@{ series = $seriesIndex; point = $pointIndex }
                        }
                    }
                }
            }
        }
        $results += [pscustomobject]@{ slide = $index; text_overflow_shape_ids = @($overflow); wrapped_data_labels = @($wrappedLabels) }
        Write-Output "Rendered slide $index."
    }
    $results | ConvertTo-Json -Depth 5 | Set-Content -LiteralPath (Join-Path $target 'render_checks.json') -Encoding UTF8
    if (@($results | Where-Object { $_.text_overflow_shape_ids.Count -gt 0 -or $_.wrapped_data_labels.Count -gt 0 }).Count -gt 0) {
        throw 'Text overflow or wrapped numeric labels detected; inspect render_checks.json.'
    }
} finally {
    if ($null -ne $deck) { $deck.Close(); [void][System.Runtime.InteropServices.Marshal]::ReleaseComObject($deck) }
    if ($null -ne $app) { $app.Quit(); [void][System.Runtime.InteropServices.Marshal]::ReleaseComObject($app) }
}
if ((Get-FileHash -LiteralPath $source -Algorithm SHA256).Hash -ne $beforeHash) {
    throw 'Rendering changed the presentation.'
}
Write-Output 'PASS: PNG previews; presentation SHA-256 unchanged.'
"""
    environment = dict(os.environ, PAPAGUN_RENDER_SOURCE=str(source),
                       PAPAGUN_RENDER_DIR=str(target),
                       PAPAGUN_RENDER_SLIDES='' if slides is None else ','.join(map(str, slides)))
    completed = subprocess.run(
        ['powershell', '-NoProfile', '-NonInteractive', '-ExecutionPolicy', 'Bypass', '-Command', script],
        env=environment, capture_output=True, text=True, encoding='utf-8', errors='replace',
        creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0),
    )
    if completed.returncode:
        raise RuntimeError('PowerPoint не завершил проверку; убедитесь, что программа установлена и PPTX доступен.')
    report = json.loads((target / 'render_checks.json').read_text(encoding='utf-8-sig'))
    if isinstance(report, dict):
        report = [report]
    require(all(not r['text_overflow_shape_ids'] and not r['wrapped_data_labels'] for r in report),
            'PowerPoint detected text overflow or wrapped numbers.')
    return report


In [ ]:
render_summary = []
if RENDER_PREVIEWS:
    if report_path is None or test_pptx_path is None:
        raise ValueError('Сначала включите GENERATE_PRESENTATIONS и выполните экспорт.')
    from IPython.display import Image, display
    with TemporaryDirectory(prefix='papagun-preview-') as preview_directory:
        for name, path in [('Итоговая', report_path), ('Тестовая', test_pptx_path)]:
            preview_path = Path(preview_directory) / name
            results = render_presentation(path, preview_path)
            for result in results:
                render_summary.append({'Презентация': name, **result})
                display(Image(filename=str(preview_path / f"slide{result['slide']}.png")))
